# CS406 — SAGE-STR v4 · Kaggle Run All (experimental)

**Kaggle:** enable **GPU** and **Internet** (first run), then **Run All**. Outputs (CSV, report, ZIP) go to `/kaggle/working/SAGE_STR_V4_GPU_RESULTS/`; critical tables are also printed in notebook. **No Google Colab download calls.**

### What is new vs v3?

- **Frozen official SVTRv2-T** and identical v3 proposal generators; compute unchanged SVTRv2-T, Geometry v3, Unified v1 controls.
- **Contextual candidate comparison:** scores use original/reference/candidate differences **and both CW/CCW proposals together**, including confidence margins, agreement, and predicted-length discrepancies. Feature use is GT-free at inference.
- **Two selectors, one GPU pass:** retrain **SAGE v3 (control)** and **SAGE v4** on the **same locked split**, then evaluate the same samples.
- **Cohort-invariant, identity-based hash split**: train 44%, calibration 11%, tune 11%, audit 14%, eval 20% assigned before sampling. Changing sample limits does not reclassify an existing image's split.
- **Three-state gain/harm/neither**, CER-aware utility, fixed computation cost, source-aware tune constraints, audit, deterministic replay, and full per-candidate forensic CSV.
- **No claim of held-out independent generalization:** LTB, IIIT5K, CUTE80, and Union14M have been repeatedly examined during development. A newly untouched benchmark is still required.

**IMPORTANT:** Kaggle Internet ON is needed for the first download unless paths in settings point to locally attached datasets/checkpoint; prefer **GPU T4×2 or P100**. Initial run defaults to a pilot. `MAX_LTB=0`, `MAX_EXTERNAL=0` means full cohort, not necessarily independent test.


In [ ]:
from pathlib import Path
import os,sys,json,hashlib,time,subprocess,importlib.util,re,io,zipfile,tarfile
import numpy as np
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
WORK.mkdir(parents=True,exist_ok=True);os.chdir(WORK)
OUT=WORK/'SAGE_STR_V4_GPU_RESULTS';OUT.mkdir(parents=True,exist_ok=True)
REPO=WORK/'OpenOCR'
OPENOCR_REV='1ccfc6ee6161f7133b192e16af3a9265273997ba'
CKPT_ID='1xJ10gOQphZjEgawXsgsadkXaZa0BIDN7'
CFG_ID='1a-NRSh9aoYpdtwNgctF7a5CbEHcTA71a'
EXPECTED_CKPT_SHA='684f23bc3a2b1f724cadfd275e1a185456dd51b5e56c8830780d1998e90f77f6'
EXPECTED_CFG_SHA='f1ab35a0f5e44d3c24c940cdf1a07404068ad9191bad9fe232ad4fcf38bbbe29'
CKPT_OVERRIDE=None
CFG_OVERRIDE=None
DATASET_ROOT_OVERRIDE=None
LTB_OVERRIDE=None
HF_DATA='topdu/OpenOCR-Data'
DATASET_NAMES=['u14m/multi_oriented','u14m/multi_words','evaluation/IIIT5k','evaluation/CUTE80']
MAX_EXTERNAL=256
MAX_LTB=1200
IMAGE_BATCH=2
AUTO_DOWNLOAD_RESULTS=False
RESUME=True
assert MAX_EXTERNAL>=0 and MAX_LTB>=0
print('SAGE v4 contextual Kaggle evaluation. LTB limit:',MAX_LTB,' external limit:',MAX_EXTERNAL,' (0=full).')
assert OUT.resolve().is_relative_to(WORK.resolve())
print("KAGGLE OUTPUT DIR:",OUT)


In [ ]:
# 1. Setup: pinned OpenOCR and minimum requirements
import subprocess, importlib.util, re, numpy as np, torch

def call(args):
    subprocess.run(args, check=True)
if not (REPO/'.git').is_dir():
    call(['git','clone','-q','https://github.com/Topdu/OpenOCR.git',str(REPO)])
call(['git','-C',str(REPO),'checkout','--force',OPENOCR_REV])
packages = [('rapidfuzz','rapidfuzz'),('gdown','gdown'),('huggingface_hub','huggingface_hub'),
            ('pandas','pandas'),('PIL','pillow'),('cv2','opencv-python-headless'),('yaml','pyyaml'),
            ('imgaug','imgaug'),('lmdb','lmdb'),('tabulate','tabulate')]
need = [pkg for mod,pkg in packages if importlib.util.find_spec(mod) is None]
if need:call([sys.executable,'-m','pip','install','-q',*need])
# OpenOCR pinned legacy imgaug may reference removed NumPy 2 aliases. Keep same compatibility patch
# used in the previously successful GAWS notebook.
spec=importlib.util.find_spec('imgaug')
if spec and spec.origin:
    substitutions=[(r'np\.sctypes\["float"\]','[np.float16,np.float32,np.float64]'),
                   (r'np\.sctypes\["int"\]','[np.int8,np.int16,np.int32,np.int64]'),
                   (r'np\.sctypes\["uint"\]','[np.uint8,np.uint16,np.uint32,np.uint64]'),
                   (r'np\.complex(?![_0-9A-Za-z])','np.complex128'),
                   (r'np\.bool(?![_0-9A-Za-z])','np.bool_'),
                   (r'np\.float(?![_0-9A-Za-z])','np.float64'),
                   (r'np\.int(?![_0-9A-Za-z])','np.int64')]
    for f in Path(spec.origin).parent.rglob('*.py'):
        try:
            raw=f.read_text(); upd=raw
            for pat,rep in substitutions: upd=re.sub(pat,rep,upd)
            if raw!=upd: f.write_text(upd)
        except (PermissionError, UnicodeDecodeError): pass
p=REPO/'openrec/preprocess/cam_label_encode.py'
if p.is_file():
    old=p.read_text();new=old.replace('pygame.locals.SRCALPHA','pygame.SRCALPHA')
    if old!=new:p.write_text(new)
sys.path.insert(0,str(REPO))
try:
    from tools.engine.config import Config
    from openrec.postprocess import build_post_process
    from openrec.modeling import build_model
except Exception as exc:
    print('OpenOCR dependency fallback:',repr(exc))
    call([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements.txt')])
    # Newly installed imgaug might require patch; re-execute compatibility replacements above if needed.
    spec=importlib.util.find_spec('imgaug')
    if spec and spec.origin:
        for f in Path(spec.origin).parent.rglob('*.py'):
            try:
                raw=f.read_text(); upd=raw
                for pat,rep in substitutions:upd=re.sub(pat,rep,upd)
                if upd!=raw:f.write_text(upd)
            except (PermissionError,UnicodeDecodeError):pass
    from tools.engine.config import Config
    from openrec.postprocess import build_post_process
    from openrec.modeling import build_model
import gdown, pandas as pd
from huggingface_hub import hf_hub_download
from PIL import Image
assert torch.cuda.is_available(), 'Enable Kaggle GPU in Notebook Settings (GPU T4 or P100).'
DEVICE='cuda'
print('OpenOCR pinned:',OPENOCR_REV,' GPU:',torch.cuda.get_device_name(0))

In [ ]:
%%writefile longtext_final.py
"""Long-text STR inference/benchmark: exact LTB Stitch v2 geometry and overlap algorithm.

Model: official SVTRv2-T OpenOCR CTC branch (strict checkpoint/config parity checks).
Methods: full12, full64, stitch_v2, geom_ctc. All share one checkpoint.
Original Stitch v2 crops the ORIGINAL PIL image, then resizes each crop with MSR.
No training, no GT-guided inference, no inference-parameter tuning on LTB.

NOTE: LTB protocol normalizes ASCII alphanumerics to lowercase and strips punctuation.
This is NOT a general-purpose punctuation-preserving OCR output pipeline.
"""
from __future__ import annotations

import math
import os
import re
import sys
import time
from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from rapidfuzz.distance import Levenshtein


# ======================= PINNED LEGACY PROCEDURE ==========================
# The two functions below are copied verbatim (algorithm) from user LTB.ipynb,
# cell G2. Do not silently 'improve' coefficients without separate validation.

def make_chunks(img, chunk_ar, overlap):
    w, h = img.size; cw = int(round(h * chunk_ar))
    if w <= cw * 1.2: return [(img, 0, w)]
    stride = max(1, int(cw * (1 - overlap))); xs = list(range(0, w - cw + 1, stride))
    if xs[-1] + cw < w: xs.append(w - cw)
    return [(img.crop((x, 0, x + cw, h)), x, x + cw) for x in xs]


def stitch_v2(parts, spans, tol=0.34):
    out, miss = parts[0], 0
    for j in range(1, len(parts)):
        b = parts[j]; wb = max(spans[j][1] - spans[j][0], 1); exp = max(0, spans[j - 1][1] - spans[j][0]) / wb * len(b)
        if exp < 0.5 or not b or not out: out += b; continue
        Lmax = min(len(out), len(b), int(round(exp * 1.6)) + 2); best = None
        for L in range(1, Lmax + 1):
            d = Levenshtein.distance(out[-L:], b[:L]); sc = L - 2 * d - 0.5 * abs(L - exp)
            if best is None or sc > best[0]: best = (sc, L, d)
        sc, L, d = best
        if sc <= 0 or d > tol * L: miss += 1; L = min(int(round(exp)), len(b)); out += b[L:]
        else: h = L // 2; out = out[:len(out) - L + h] + b[h:]
    return out, miss

# Legacy values selected among eight configurations in original LTB study.
# They are EMPIRICAL LOCKED CONFIG, not theoretic constants or a new grid search.
LEGACY_CHUNK_AR = 6.0
LEGACY_OVERLAP = 0.5


def normalize_ltb(s: str) -> str:
    return re.sub(r'[^0-9a-z]', '', str(s).lower())


def target_size_ltb(w: int, h: int, max_ratio: int = 12):
    """Original LTB.ipynb 'msr_round' target_size, copied geometry exactly."""
    if w <= 0 or h <= 0:
        raise ValueError('Invalid image width/height')
    base = [(64, 64), (96, 48), (112, 40), (128, 32)]
    r = min(max(1, round(w / h)), max_ratio)
    return base[r - 1] if r <= 4 else (32 * r, 32)


def preprocess_ltb(img: Image.Image, max_ratio: int = 12) -> torch.Tensor:
    """Original PIL bicubic -> RGB numpy -> [-1,1]."""
    w, h = img.size
    W, H = target_size_ltb(w, h, max_ratio)
    bicubic = Image.Resampling.BICUBIC if hasattr(Image, 'Resampling') else Image.BICUBIC
    a = np.asarray(img.convert('RGB').resize((W, H), bicubic), dtype=np.float32) / 255.
    return torch.from_numpy(((a - 0.5) / 0.5).transpose(2, 0, 1).copy())


# ======================= OFFICIAL SVTRv2-T ================================
# Identical tested RCTC extraction and strict load logic from GAWS notebook.

def rctc_logits(dec, x):
    B, C, H, W = x.shape
    x = dec.w_atten_block(x.permute(0, 2, 3, 1).reshape(-1, W, C)).reshape(B, H, W, C).permute(0, 3, 1, 2)
    x_kv = dec.fc_kv(x.flatten(2).transpose(1, 2)).reshape(B, H * W, 2, C).permute(2, 0, 3, 1)
    x_k, x_v = x_kv.unbind(0)
    tok = dec.char_token.tile([B, 1, 1])
    attn = (tok @ x_k).reshape([-1, 1, H, W])
    attn = F.softmax(attn, 2).permute(0, 3, 1, 2)
    x_v = x_v.reshape(B, C, H, W)
    feats = (attn @ x_v.permute(0, 3, 2, 1)).squeeze(2)
    return dec.fc(feats)


class CTCNet(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.transform = getattr(model, 'transform', None)
        self.encoder = model.encoder
        d = model.decoder
        self.rctc = d if hasattr(d, 'w_atten_block') else d.ctc_decoder

    def forward(self, x):
        if self.transform is not None:
            x = self.transform(x)
        return rctc_logits(self.rctc, self.encoder(x))


def load_official_svtr(repo, cfg_path, ckpt_path, device='cuda'):
    """Build official architecture, strict checkpoint check, verify extracted logits parity."""
    repo, cfg_path, ckpt_path = map(Path, (repo, cfg_path, ckpt_path))
    if not repo.is_dir() or not cfg_path.is_file() or not ckpt_path.is_file():
        raise FileNotFoundError((str(repo), str(cfg_path), str(ckpt_path)))
    sys.path.insert(0, str(repo))
    from tools.engine.config import Config
    from openrec.modeling import build_model
    from openrec.postprocess import build_post_process

    old_dir = Path.cwd()
    try:
        os.chdir(repo)  # official YAML uses repo-relative character dictionary paths
        cfg = Config(str(cfg_path)).cfg
        post = build_post_process(cfg['PostProcess'], cfg['Global'])
        channels = post.get_character_num()
        ccfg = {
            'name': 'CTCLabelDecode',
            'character_dict_path': cfg['PostProcess'].get('character_dict_path') or cfg['Global'].get('character_dict_path'),
            'use_space_char': cfg['PostProcess'].get('use_space_char', cfg['Global'].get('use_space_char', False)),
        }
        chars = list(build_post_process(ccfg, cfg['Global']).character)
        cfg['Architecture']['Decoder']['out_channels'] = channels
        model = build_model(cfg['Architecture'])
    finally:
        os.chdir(old_dir)

    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    sd = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    if not isinstance(sd, dict):
        raise RuntimeError('Invalid checkpoint format')
    sd = {k[7:] if k.startswith('module.') else k: v for k, v in sd.items()}
    ref = model.state_dict()
    missing = set(ref) - set(sd)
    extra = set(sd) - set(ref)
    wrong = [k for k in set(ref) & set(sd) if ref[k].shape != sd[k].shape]
    if missing or extra or wrong:
        raise RuntimeError(f'Checkpoint/config mismatch: missing={list(missing)[:4]} extra={list(extra)[:4]} shape={wrong[:4]}')
    model.load_state_dict(sd, strict=True)
    if not chars or str(chars[0]).lower() not in {'blank', '[blank]', '<blank>', 'blank_token'}:
        raise RuntimeError(f'CTC character index 0 must be blank: {chars[:5]}')
    model = model.to(device).eval()
    net = CTCNet(model).to(device).eval()
    if net.rctc.fc.out_features != len(chars):
        raise RuntimeError('CTC output class count differs from dictionary')
    # Check CTCNet exactly matches official RCTC pre-softmax logit.
    probe = torch.randn(1, 3, 32, 192, device=device)
    captured = []
    hook = net.rctc.fc.register_forward_hook(lambda m, arg, output: captured.append(output.detach()))
    try:
        with torch.inference_mode():
            model(probe)
            logits = net(probe)
    finally:
        hook.remove()
    if len(captured) != 2 or not torch.allclose(captured[0], logits, atol=3e-4, rtol=3e-4):
        raise RuntimeError('Official-vs-extracted RCTC logits parity failed')
    del model, probe, captured
    return net, chars


def greedy_ctc(logits: np.ndarray, chars: list[str]) -> str:
    """CTC collapse, blank=0. No blank bias, beam width or confidence threshold."""
    pred = np.asarray(logits).argmax(-1).tolist()
    out, prev = [], -1
    for c in pred:
        if c != prev and c != 0:
            out.append(chars[c])
        prev = c
    return ''.join(out)


def geometry_ctc_same_crops(logits: list[np.ndarray], spans: list[tuple[int, int]], chars: list[str]) -> str:
    """Deterministic center-ownership stitching of CTC timesteps on ORIGINAL crop coordinates.

    SAME crops, backbone, preprocessing, and forward passes as Stitch v2.
    This is a baseline, not a novel or guaranteed accurate CTC alignment algorithm.
    """
    if not logits or len(logits) != len(spans):
        raise ValueError('Need one CTC logit tensor per crop span')
    if len(logits) == 1:
        return greedy_ctc(logits[0], chars)
    centers = [(a + b) / 2 for a, b in spans]
    cuts = [-float('inf')] + [(centers[i] + centers[i+1]) / 2 for i in range(len(centers)-1)] + [float('inf')]
    pieces = []
    for j, (lg, (x0, x1)) in enumerate(zip(logits, spans)):
        t = len(lg)
        if t == 0:
            raise RuntimeError('CTC produced zero timesteps for a crop')
        x = x0 + (np.arange(t) + 0.5) * ((x1 - x0) / t)
        keep = (x >= cuts[j]) & (x < cuts[j+1])
        if not keep.any():
            raise RuntimeError(f'No owned CTC frames for window {j}')
        pieces.append(np.asarray(lg)[keep])
    return greedy_ctc(np.concatenate(pieces, axis=0), chars)


class LongTextRecognizer:
    METHODS = ('full12', 'full64', 'stitch_v2', 'geom_ctc')

    def __init__(self, net: nn.Module, chars: list[str], device: str='cuda', batch_size: int=16):
        self.net = net.to(device).eval()
        self.chars = list(chars)
        self.device = str(device)
        self.batch_size = max(1, int(batch_size))  # memory-only knob, never tuned on accuracy
        if self.chars[0].lower() not in ('blank', '[blank]', '<blank>', 'blank_token'):
            raise ValueError('CTC blank must be class 0')

    @torch.inference_mode()
    def _forward_grouped(self, pils: list[Image.Image], max_ratio: int) -> list[np.ndarray]:
        tens = [preprocess_ltb(im, max_ratio) for im in pils]
        groups = defaultdict(list)
        for i, t in enumerate(tens):
            groups[tuple(t.shape)].append(i)
        ret: list[np.ndarray | None] = [None] * len(tens)
        for shape, ids in groups.items():
            _, h, w = shape
            # Bound temporary tensor volume as image width grows; only affects memory/throughput.
            bs = max(1, min(self.batch_size, int(self.batch_size * 192 / max(w, 1))))
            for start in range(0, len(ids), bs):
                ii = ids[start:start+bs]
                x = torch.stack([tens[i] for i in ii]).to(self.device)
                logits = self.net(x).float().detach().cpu().numpy()
                if logits.ndim != 3 or logits.shape[-1] != len(self.chars):
                    raise RuntimeError(f'Unexpected CTC shape: {logits.shape}')
                for i, lg in zip(ii, logits):
                    ret[i] = lg
        if any(x is None for x in ret):
            raise RuntimeError('Missing CTC results')
        return ret  # type: ignore[return-value]

    def predict_many(self, imgs: list[Image.Image], method: str='stitch_v2', return_info: bool=False):
        if method not in self.METHODS:
            raise ValueError(f'Method must be one of {self.METHODS}')
        if not imgs:
            return []
        imgs = [im.convert('RGB') for im in imgs]
        if method.startswith('full'):
            ratio = 12 if method == 'full12' else 64
            outs = self._forward_grouped(imgs, ratio)
            result = [{'prediction': normalize_ltb(greedy_ctc(lg, self.chars)),
                       'num_windows': 1, 'stitch_misses': 0} for lg in outs]
        else:
            crops, meta = [], []
            for im in imgs:
                left = len(crops)
                parts = make_chunks(im, LEGACY_CHUNK_AR, LEGACY_OVERLAP)
                crops.extend(p for p, _, _ in parts)
                spans = [(a, b) for _, a, b in parts]
                meta.append((left, len(parts), spans))
            lgs = self._forward_grouped(crops, max_ratio=12)
            result = []
            for left, count, spans in meta:
                crop_lgs = lgs[left:left+count]
                raw = [greedy_ctc(x, self.chars) for x in crop_lgs]
                norm_parts = [normalize_ltb(s) for s in raw]
                if method == 'stitch_v2':
                    text, misses = stitch_v2(norm_parts, spans)
                else:
                    text, misses = normalize_ltb(geometry_ctc_same_crops(crop_lgs, spans, self.chars)), 0
                result.append({'prediction': text, 'num_windows': count,
                               'stitch_misses': misses, 'crop_predictions': norm_parts,
                               'crop_spans': spans})
        return result if return_info else [r['prediction'] for r in result]

    def predict(self, img: Image.Image, method: str='stitch_v2', return_info: bool=False):
        return self.predict_many([img], method, return_info)[0]


def score_predictions(df):
    """df columns: id, gt, pred, optional bucket. Output exact-line acc & CER/S/D/I by bucket."""
    import pandas as pd
    rows = []
    for row in df.itertuples():
        gt = normalize_ltb(row.gt)
        pr = normalize_ltb(row.pred)
        ops = Levenshtein.editops(gt, pr)
        s = sum(o.tag == 'replace' for o in ops)
        d = sum(o.tag == 'delete' for o in ops)
        ins = sum(o.tag == 'insert' for o in ops)
        bucket = '26-35' if len(gt) <= 35 else ('36-55' if len(gt) <= 55 else '56+')
        rows.append({'id': row.id, 'gt': gt, 'pred': pr, 'bucket': bucket,
                     'correct': int(gt == pr), 'L': len(gt), 'S': s, 'D': d, 'I': ins})
    out = pd.DataFrame(rows)
    return out


def metric_table(df):
    """A-Avg only if ALL three LTB buckets are present."""
    import pandas as pd
    rows=[]
    for b in ['26-35', '36-55', '56+', 'ALL']:
        sub = df if b == 'ALL' else df[df.bucket == b]
        if len(sub) == 0:
            continue
        L = int(sub.L.sum())
        rows.append({'bucket': b, 'n': len(sub), 'accuracy': 100*sub.correct.mean(),
                     'CER': 100*(sub.S.sum()+sub.D.sum()+sub.I.sum())/max(L,1),
                     'S%': 100*sub.S.sum()/max(L,1),
                     'D%': 100*sub.D.sum()/max(L,1),
                     'I%': 100*sub.I.sum()/max(L,1)})
    present = set(df.bucket)
    if {'26-35','36-55','56+'} <= present:
        rows.append({'bucket': 'A-Avg', 'n':len(df),
                     'accuracy': float(np.mean([x['accuracy'] for x in rows if x['bucket'] in ('26-35','36-55','56+')]))})
    return pd.DataFrame(rows)


def run_regression_tests():
    # Three tests copied from original LTB; plus frame ownership and CTC blank semantics.
    sp = [(0, 100), (50, 150)]
    assert stitch_v2(['abcdefgh', 'efghijkl'], sp)[0] == 'abcdefghijkl'
    assert stitch_v2(['abcdefgh', 'xxxxyyyy'], [(0, 100), (100, 200)])[0] == 'abcdefghxxxxyyyy'
    im = Image.new('RGB',(170,20),'white')
    cs = make_chunks(im, LEGACY_CHUNK_AR, LEGACY_OVERLAP)
    assert [(a,b) for _,a,b in cs] == [(0,120),(50,170)]
    assert len(make_chunks(Image.new('RGB',(100,20)),LEGACY_CHUNK_AR,LEGACY_OVERLAP)) == 1
    # blank=0; two letters same token separated by blank must survive collapse.
    chars = ['blank','a','b']
    logits = np.array([[0,5,0],[0,5,0],[5,0,0],[0,5,0]],dtype=np.float32)
    assert greedy_ctc(logits,chars) == 'aa'
    one = np.array([[6,0,0],[0,8,0],[6,0,0],[0,0,8]],dtype=np.float32)
    two = np.array([[0,8,0],[6,0,0],[0,0,8],[6,0,0]],dtype=np.float32)
    assert geometry_ctc_same_crops([one,two],[(0,100),(50,150)],chars) == 'ab'
    sizes = [target_size_ltb(100,20), target_size_ltb(300,20), target_size_ltb(1500,20,12)]
    assert sizes == [(160,32),(384,32),(384,32)],sizes
    return True


if __name__ == '__main__':
    run_regression_tests()
    print('PASS: deterministic geometry/stitch/CTC/multi-window tests')

In [ ]:
%%writefile longtext_v3.py
"""Research candidate: Stitch v3 evidence-aware post-processing for SVTRv2 CTC.

NOT a validated improvement. Same *original-image* crops, same model and same
inference passes as Stitch v2. We change only the text-merge stage.

Frozen original Stitch v2 is imported from longtext_final.py. Three ablations:
    v3_conf: preserve v2 overlap selection; on *aligned single-character substitutions*
             choose the higher CTC event confidence rather than v2's fixed midpoint.
    v3_geom: preserve v2 successful matches; if v2 rejects a match, select by
             original-image geometry at the midpoint between the crop centres.
    v3_both: both rules together.

No new tuned hyperparameters. Confidence is a diagnostic proxy (not calibrated),
frame-to-pixel locations are approximate, and empirical gains are NOT guaranteed.
This is a coursework research prototype, not a new SOTA or publication claim.
"""
from __future__ import annotations

from dataclasses import dataclass
from collections import defaultdict
from typing import Sequence
import time

import numpy as np
from PIL import Image
from rapidfuzz.distance import Levenshtein

from longtext_final import (
    LongTextRecognizer, make_chunks, stitch_v2, normalize_ltb,
    LEGACY_CHUNK_AR, LEGACY_OVERLAP, greedy_ctc, geometry_ctc_same_crops,
)


@dataclass(frozen=True)
class CharEvidence:
    char: str
    confidence: float
    x: float


def ctc_events(logits: np.ndarray, chars: Sequence[str], x0: int, x1: int) -> list[CharEvidence]:
    """Convert greedy CTC character runs to confidence and approximate global x.

    For each nonblank run, its symbol probability is measured at the most
    confident timestep for that symbol; its position is that timestep's centre.
    This is not forced alignment and the position may differ from glyph centre.
    """
    v = np.asarray(logits, dtype=np.float64)
    if v.ndim != 2 or v.shape[1] != len(chars) or v.shape[0] < 1:
        raise ValueError(f'Expected nonempty [T,C] logits, got {v.shape}')
    z = v - np.max(v, axis=1, keepdims=True)
    expz = np.exp(z)
    p = expz / np.sum(expz, axis=1, keepdims=True)
    ids = np.argmax(v, axis=1)
    events = []
    t = 0
    while t < len(ids):
        j = t + 1
        while j < len(ids) and ids[j] == ids[t]:
            j += 1
        cls = int(ids[t])
        if cls != 0:
            label = normalize_ltb(str(chars[cls]))
            if len(label) > 1:
                raise ValueError(f'CTC label {chars[cls]!r} normalizes to multiple characters')
            if label:
                peak = t + int(np.argmax(p[t:j, cls]))
                x = float(x0 + (peak + 0.5) * ((x1 - x0) / len(ids)))
                events.append(CharEvidence(label, float(p[peak, cls]), x))
        t = j
    plain = normalize_ltb(greedy_ctc(v, list(chars)))
    if ''.join(e.char for e in events) != plain:
        raise AssertionError('Character-level evidence disagrees with greedy CTC output')
    return events


def _v2_match_info(out: str, b: str, prev_span, next_span, tol=0.34):
    """Exact legacy v2 scoring and acceptance; no change to existing thresholds."""
    wb = max(next_span[1] - next_span[0], 1)
    exp = max(0, prev_span[1] - next_span[0]) / wb * len(b)
    if exp < 0.5 or not b or not out:
        return {'mode': 'concat', 'exp': exp}
    lmax = min(len(out), len(b), int(round(exp * 1.6)) + 2)
    best = None
    for L in range(1, lmax + 1):
        d = Levenshtein.distance(out[-L:], b[:L])
        sc = L - 2*d - 0.5*abs(L-exp)
        if best is None or sc > best[0]:
            best = (sc, L, d)
    if best is None:
        return {'mode':'concat','exp':exp}
    sc,L,d = best
    return {
        'mode': ('reject' if sc<=0 or d>tol*L else 'accept'),
        'L': L, 'd': d, 'sc': sc, 'exp': exp,
    }


def stitch_evidence(
    chunks: Sequence[Sequence[CharEvidence]],
    spans: Sequence[tuple[int, int]],
    *,
    use_confidence: bool=True,
    geometric_fallback: bool=True,
    return_info: bool=False,
):
    """Stitch v3 candidate. Only alters v2 on disagreement or rejected overlap.

    No extra inference, no new learned weights, no threshold fitted to test.
    Comparison against raw v2 is essential; only adopt if robustly better.
    """
    if not chunks or len(chunks) != len(spans):
        raise ValueError('Need nonempty chunks and one span per chunk')
    spans = [tuple(map(int,s)) for s in spans]
    out = list(chunks[0])
    misses = 0
    changes = 0
    geom_misses = 0
    ambiguous = 0
    for j in range(1, len(chunks)):
        nxt = list(chunks[j])
        a = ''.join(e.char for e in out)
        b = ''.join(e.char for e in nxt)
        match = _v2_match_info(a,b,spans[j-1],spans[j])
        mode = match['mode']
        if mode == 'concat':
            out += nxt
            continue
        L = match['L']
        if mode == 'reject':
            misses += 1
            if geometric_fallback:
                # Midpoint between crop centres; lies inside their overlap for
                # equal-size windows. No predicted labels or GT used to route.
                prev_center = (spans[j-1][0] + spans[j-1][1]) / 2
                next_center = (spans[j][0] + spans[j][1]) / 2
                cut_x = (prev_center + next_center) / 2
                out = [e for e in out if e.x < cut_x] + [e for e in nxt if e.x >= cut_x]
                geom_misses += 1
            else:
                strip = min(int(round(match['exp'])),len(nxt))
                out += nxt[strip:]
            continue
        # This was an accepted legacy V2 match. Preserve it unless competing
        # CTC events are positionally aligned 1:1 with a substitution.
        h = L // 2
        left = out[-L:]
        right = nxt[:L]
        if use_confidence and match['d']:
            ops = Levenshtein.editops(''.join(x.char for x in left),
                                      ''.join(x.char for x in right))
            # Only same-index substitutions: no artificial re-indexing of
            # deletion/insertion or repeated-token alignments.
            safe = all(op.tag=='replace' and op.src_pos==op.dest_pos for op in ops)
            if safe:
                consensus = []
                for i in range(L):
                    aa, bb = left[i],right[i]
                    default = aa if i < h else bb
                    if aa.char == bb.char:
                        consensus.append(default)
                    elif aa.confidence > bb.confidence:
                        consensus.append(aa)
                        changes += int(default.char != aa.char)
                    elif bb.confidence > aa.confidence:
                        consensus.append(bb)
                        changes += int(default.char != bb.char)
                    else:
                        consensus.append(default)
                out = out[:-L] + consensus + nxt[L:]
                continue
            ambiguous += 1
        out = out[:-L+h] + nxt[h:]
    text = ''.join(e.char for e in out)
    if return_info:
        return {'prediction':text, 'stitch_misses':misses,
                'confidence_char_flips':changes, 'geometry_fallbacks':geom_misses,
                'unalignable_overlap':ambiguous}
    return text


class EvidenceLongTextRecognizer(LongTextRecognizer):
    """Reuses original crop and GPU logits for all v2/v3/geometry comparisons."""
    CROP_METHODS = ('stitch_v2','v3_conf','v3_geom','v3_both','geom_ctc')
    METHODS = LongTextRecognizer.METHODS + tuple(x for x in CROP_METHODS if x not in LongTextRecognizer.METHODS)

    def predict_all_cropped(self, imgs: list[Image.Image], return_info=False):
        imgs = [im.convert('RGB') for im in imgs]
        all_crops=[]; meta=[]
        for im in imgs:
            i = len(all_crops)
            parts=make_chunks(im,LEGACY_CHUNK_AR,LEGACY_OVERLAP)
            spans=[(a,b) for _,a,b in parts]
            all_crops.extend(p for p,_,_ in parts)
            meta.append((i,len(parts),spans))
        t0=time.perf_counter()
        all_lgs=self._forward_grouped(all_crops, max_ratio=12) if all_crops else []
        forward_seconds=time.perf_counter()-t0
        res={m:[] for m in self.CROP_METHODS}
        cpu_seconds=defaultdict(float)
        evidence_seconds=0.0
        for start,n,spans in meta:
            lgs=all_lgs[start:start+n]
            t1=time.perf_counter()
            evs=[ctc_events(lg,self.chars,a,b) for lg,(a,b) in zip(lgs,spans)]
            parts=[''.join(e.char for e in ev) for ev in evs]
            evidence_seconds+=time.perf_counter()-t1
            t1=time.perf_counter()
            basic,miss=stitch_v2(parts,spans)
            result={'stitch_v2':{'prediction':basic,'stitch_misses':miss}}
            cpu_seconds['stitch_v2']+=time.perf_counter()-t1
            for method,cc,gf in [('v3_conf',True,False),('v3_geom',False,True),('v3_both',True,True)]:
                t1=time.perf_counter()
                result[method]=stitch_evidence(evs,spans,use_confidence=cc,
                                              geometric_fallback=gf,return_info=True)
                cpu_seconds[method]+=time.perf_counter()-t1
            t1=time.perf_counter()
            result['geom_ctc']={'prediction':normalize_ltb(geometry_ctc_same_crops(lgs,spans,self.chars)),
                                'stitch_misses':0}
            cpu_seconds['geom_ctc']+=time.perf_counter()-t1
            for method in self.CROP_METHODS:
                d=dict(result[method]);d['num_windows']=n
                if return_info:
                    d['crop_predictions']=parts
                    d['crop_spans']=spans
                res[method].append(d)
            # Core parity: method with neither intervention is exactly v2.
            if stitch_evidence(evs,spans,use_confidence=False,geometric_fallback=False) != basic:
                raise AssertionError('Stitch v3 core does not reproduce legacy v2')
        return res, {'forward_seconds':forward_seconds,'evidence_seconds':evidence_seconds,
                     'cpu_seconds':dict(cpu_seconds),
                     'crops':len(all_crops), 'images':len(imgs)}

    def predict_many(self,imgs,method='stitch_v2',return_info=False):
        if method in self.CROP_METHODS:
            res,_ = self.predict_all_cropped(imgs,return_info)
            return res[method] if return_info else [x['prediction'] for x in res[method]]
        return super().predict_many(imgs,method,return_info)


def run_regression_tests_v3():
    from longtext_final import run_regression_tests
    assert run_regression_tests()
    chars=['blank','a','b','c','d']
    def logits(seq):
        ans=[]
        for idx in seq:
            row=np.full(len(chars), -8.,dtype=float)
            row[idx]=5.
            ans.append(row)
        return np.asarray(ans)
    lg=logits([1,1,0,1,2,0,2])
    e=ctc_events(lg,chars,10,80)
    assert ''.join(x.char for x in e)=='aabb' and all(10<=x.x<=80 for x in e)
    # Exact agreement is always preserved, no matter which ablation is on.
    p=['abcdef','cdefgh','efghij']
    spans=[(0,100),(50,150),(100,200)]
    evs=[[CharEvidence(c,.9,x0+(i+.5)*100/len(s)) for i,c in enumerate(s)]
          for s,(x0,x1) in zip(p,spans)]
    base=stitch_v2(p,spans)[0]
    for conf in (False,True):
        for geom in (False,True):
            ans=stitch_evidence(evs,spans,use_confidence=conf,geometric_fallback=geom)
            assert ans == base,(conf,geom,ans,base)
    # Exhaustively verify parity with original v2 for all random characters
    # when both interventions are disabled.
    rng=np.random.default_rng(14)
    alphabet='abcde'
    for i in range(200):
        k=int(rng.integers(1,7))
        ss=[''.join(rng.choice(list(alphabet),size=int(rng.integers(0,18)))) for j in range(k)]
        sp=[(j*50,j*50+100) for j in range(k)]
        evs=[[CharEvidence(c,.7,sp[j][0]+(z+.5)*100/max(len(s),1)) for z,c in enumerate(s)]
             for j,s in enumerate(ss)]
        a=stitch_v2(ss,sp)[0]
        b=stitch_evidence(evs,sp,use_confidence=False,geometric_fallback=False)
        assert a==b,(ss,sp,a,b)
    # Repeated letters remain separate events when separated by blank.
    assert ctc_events(logits([1,0,1]),chars,0,30)[0].char == 'a'
    assert ''.join(x.char for x in ctc_events(logits([1,0,1]),chars,0,30))=='aa'
    # Detect a real confidence substitution correction in an accepted overlap.
    a=list('abcxef');b=list('cdefgh');sp=[(0,100),(50,150)]
    ae=[CharEvidence(c, .02 if c=='x' else .8, (j+.5)*100/len(a)) for j,c in enumerate(a)]
    be=[CharEvidence(c, .99 if c=='d' else .8, 50+(j+.5)*100/len(b)) for j,c in enumerate(b)]
    base=stitch_v2([''.join(a),''.join(b)],sp)[0]
    newer=stitch_evidence([ae,be],sp,use_confidence=True,geometric_fallback=False)
    assert base!=newer and 'd' in newer,(base,newer)
    return {'passed':True,'toy_correction':{'v2':base,'v3_conf':newer},'random_parity_cases':200}


if __name__ == '__main__':
    print(run_regression_tests_v3())

In [ ]:
%%writefile orientation_probe.py
"""GT-blind orientation probe. Frozen original crop-before-resize + v3 geometry.

Never call this module with label text. All selector criteria use predictions,
CTC event geometry and confidence only. Scores are exploratory, not calibrated.
"""
from __future__ import annotations
from collections import defaultdict
from difflib import SequenceMatcher
import math,time
import numpy as np
import torch
from PIL import Image
import longtext_final as LT
import longtext_v3 as V3

METHODS=('original','cw_v3','ccw_v3','cw_v2','ccw_v2','cw_geom_ctc','ccw_geom_ctc','select_overlap','select_charconf')

def rotated(im,which):
    if which == 'original':return im.convert('RGB')
    if which == 'cw':return im.convert('RGB').transpose(Image.Transpose.ROTATE_270)
    if which == 'ccw':return im.convert('RGB').transpose(Image.Transpose.ROTATE_90)
    raise ValueError(which)

def overlap_agreement(event_lists,spans):
    match=0.;denom=0
    for i in range(len(spans)-1):
        left,right=event_lists[i],event_lists[i+1]
        a=max(spans[i][0],spans[i+1][0]);b=min(spans[i][1],spans[i+1][1])
        if b<=a:continue
        s1=''.join(e.char for e in left if a<=e.x<=b)
        s2=''.join(e.char for e in right if a<=e.x<=b)
        # Weight by shared recognized event count; empty/empty gives no false credit
        c=len(s1)+len(s2)
        if c:
            match+=c*SequenceMatcher(None,s1,s2,autojunk=False).ratio()
            denom+=c
    return float(match/denom) if denom else 0.0

def event_confidence(events):
    vals=[e.confidence for sub in events for e in sub]
    return float(np.mean(vals)) if vals else 0.0

def choose_orientation(a,b,kind='overlap'):
    # Positive direction names are literal rotations; no ground-truth, no label length
    if kind=='overlap':
        key_a=(a['agreement'],a['confidence'],len(a['pred']))
        key_b=(b['agreement'],b['confidence'],len(b['pred']))
    elif kind=='charconf':
        key_a=(a['confidence'],a['agreement'],len(a['pred']))
        key_b=(b['confidence'],b['agreement'],len(b['pred']))
    else:raise ValueError(kind)
    return ('cw',a) if key_a>=key_b else ('ccw',b)

class OrientationProbe:
    def __init__(self,net,chars,device='cuda',batch_size=2):
        self.core=LT.LongTextRecognizer(net,chars,device=device,batch_size=batch_size)
        self.chars=list(chars)

    def infer_batch(self,images):
        images=[im.convert('RGB') for im in images]
        # Original is 1-crop for this cohort. Orientation candidates use exact
        # make_chunks(raw pixels) THEN individual resized crop.
        allparts=[];meta=[]
        for n,im in enumerate(images):
            for angle in ('original','cw','ccw'):
                candidate=rotated(im,angle)
                parts=LT.make_chunks(candidate,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
                start=len(allparts)
                allparts.extend(p for p,_,_ in parts)
                meta.append((n,angle,start,len(parts),[(x0,x1) for _,x0,x1 in parts]))
        t0=time.perf_counter()
        logits=self.core._forward_grouped(allparts,max_ratio=12)
        gpu_s=time.perf_counter()-t0
        record=[{} for _ in images]
        for n,angle,start,count,spans in meta:
            cur=logits[start:start+count]
            events=[V3.ctc_events(lg,self.chars,a,b) for lg,(a,b) in zip(cur,spans)]
            v3=V3.stitch_evidence(events,spans,use_confidence=False,geometric_fallback=True)
            parts=[''.join(e.char for e in ev) for ev in events]
            v2,_=LT.stitch_v2(parts,spans)
            geom=LT.normalize_ltb(LT.geometry_ctc_same_crops(cur,spans,self.chars))
            result={'pred':v3,'v2':v2,'geom_ctc':geom,'n_crops':count,
                    'agreement':overlap_agreement(events,spans),
                    'confidence':event_confidence(events),'T_sum':sum(len(z) for z in cur)}
            record[n][angle]=result
        outputs=[]
        for r in record:
            cw=r['cw'];ccw=r['ccw']
            a=choose_orientation(cw,ccw,'overlap')
            b=choose_orientation(cw,ccw,'charconf')
            d={'original':r['original']['pred'],'cw_v3':cw['pred'],'ccw_v3':ccw['pred'],
               'cw_v2':cw['v2'],'ccw_v2':ccw['v2'],
               'cw_geom_ctc':cw['geom_ctc'],'ccw_geom_ctc':ccw['geom_ctc'],
               'select_overlap':a[1]['pred'],'select_charconf':b[1]['pred'],
               'overlap_choice':a[0],'charconf_choice':b[0],
               'cw_agreement':cw['agreement'],'ccw_agreement':ccw['agreement'],
               'cw_confidence':cw['confidence'],'ccw_confidence':ccw['confidence'],
               'cw_n_crops':cw['n_crops'],'ccw_n_crops':ccw['n_crops'],
               'original_T':r['original']['T_sum']}
            outputs.append(d)
        return outputs,{'num_tensors':len(allparts),'gpu_forward_seconds':gpu_s}

def run_cpu_tests():
    from PIL import Image
    class Dummy(torch.nn.Module):
        def forward(self,x):
            b,_,h,w=x.shape
            v=torch.zeros(b,w//4,3,device=x.device);v[:,:,0]=2.;v[:,(w//8):((w//8)+2),1]=3.
            return v
    im=Image.new('RGB',(18,210),'white')
    assert rotated(im,'cw').size==(210,18) and rotated(im,'ccw').size==(210,18)
    model=OrientationProbe(Dummy(),['blank','a','b'],device='cpu',batch_size=2)
    x,perf=model.infer_batch([im,Image.new('RGB',(31,220),'white')])
    assert len(x)==2 and all(set(METHODS).issubset(z.keys()) for z in x)
    assert all(z['overlap_choice'] in ('cw','ccw') for z in x)
    assert all(isinstance(z[k],str) for z in x for k in METHODS)
    assert any(z['cw_n_crops']>=2 for z in x)
    assert perf['num_tensors']>=10
    # Selector cannot access GT because it accepts only orientation dictionaries
    assert choose_orientation({'pred':'a','agreement':0.8,'confidence':0.3},
                              {'pred':'b','agreement':0.5,'confidence':0.9},'overlap')[0]=='cw'
    return {'passed':True,'methods':list(METHODS),'dummy_tensors':perf['num_tensors']}

In [ ]:
%%writefile density_probe.py
"""Input-resolution/forced-crop probe for high-density LTB lines.

All transformations GT-blind and predetermined. No model training or test tuning.
"""
from __future__ import annotations
from collections import defaultdict
from PIL import Image
import numpy as np
import torch
import time
from typing import Sequence
import longtext_final as LT
import longtext_v3 as V3

METHODS = ('baseline','width_x2','width_x3','force_2','force_3','force_3_width_x3')
MAX_INPUT_W = 768


def resized_tensor(img: Image.Image, factor: int) -> torch.Tensor:
    """Original MSR when factor=1; direct resize raw pixels (no double interpolation).

    factor>1 enlarges *horizontal* input dimension while keeping MSR height,
    a diagnostic OOD intervention on the frozen checkpoint.
    """
    if factor == 1:
        return LT.preprocess_ltb(img,max_ratio=12)
    w, h = LT.target_size_ltb(*img.size,max_ratio=12)
    w2 = int(min(MAX_INPUT_W, 4*round(w*factor/4)))
    if w2 <= w: raise ValueError('Expected an expanded width')
    mode = Image.Resampling.BICUBIC if hasattr(Image,'Resampling') else Image.BICUBIC
    a = np.asarray(img.convert('RGB').resize((w2,h),mode),dtype=np.float32)/255.
    return torch.from_numpy(((a-0.5)/0.5).transpose(2,0,1).copy())


def forced_crops(img: Image.Image, n: int):
    """Overlapping 2- or 3-view raw-image tiling, no GT or recognition output."""
    if n not in (2,3): raise ValueError('Only 2 and 3 fixed raw-image views')
    w,h=img.size
    if w < 8: return [(img,0,w)]  # no-op for truly narrow images; log number of crops
    cw=min(w-1,max(4,int(round(w*(0.65 if n==2 else 0.52)))))
    starts = [0,w-cw] if n==2 else [0,int(round((w-cw)/2)),w-cw]
    if len(set(starts)) != n: raise ValueError('Degenerate forced-crop geometry')
    assert starts[0]==0 and starts[-1]+cw == w
    return [(img.crop((x,0,x+cw,h)),x,x+cw) for x in starts]


class Probe:
    def __init__(self,net,chars,device='cuda',batch_size=8):
        self.net=net.to(device).eval();self.chars=list(chars)
        self.device=device;self.batch_size=batch_size
        if not self.chars or str(self.chars[0]).lower() not in ('blank','[blank]','<blank>','blank_token'):
            raise ValueError('Expected blank class zero')

    @torch.inference_mode()
    def _forward_tensors(self,tensors: Sequence[torch.Tensor]) -> list[np.ndarray]:
        groups=defaultdict(list)
        for i,t in enumerate(tensors):groups[tuple(t.shape)].append(i)
        ret=[None]*len(tensors)
        for shape,ids in groups.items():
            _,h,w=shape
            volume=max(1,w*h)
            bs=max(1,min(self.batch_size,int(self.batch_size*192*32/volume)))
            for start in range(0,len(ids),bs):
                ii=ids[start:start+bs]
                xx=torch.stack([tensors[i] for i in ii],dim=0).to(self.device)
                raw=self.net(xx).detach().float().cpu().numpy()
                assert raw.ndim==3 and raw.shape[2]==len(self.chars),(raw.shape,len(self.chars))
                for i,x in zip(ii,raw):ret[i]=x
        assert all(x is not None for x in ret)
        return ret

    def batch(self, imgs: Sequence[Image.Image]):
        imgs=[im.convert('RGB') for im in imgs]
        inputs=[];mapping=[]
        for i,im in enumerate(imgs):
            for m in METHODS:
                if m=='baseline': parts=[(im,0,im.width)]; factor=1
                elif m=='width_x2':parts=[(im,0,im.width)]; factor=2
                elif m=='width_x3':parts=[(im,0,im.width)]; factor=3
                elif m=='force_2':parts=forced_crops(im,2);factor=1
                elif m=='force_3':parts=forced_crops(im,3);factor=1
                elif m=='force_3_width_x3':parts=forced_crops(im,3);factor=3
                else:raise AssertionError(m)
                start=len(inputs)
                inputs.extend(resized_tensor(p,factor) for p,_,_ in parts)
                spans=[(a,b) for _,a,b in parts]
                mapping.append((i,m,start,len(parts),spans))
        t0=time.perf_counter();logits=self._forward_tensors(inputs)
        if str(self.device).startswith('cuda'):torch.cuda.synchronize()
        inference_s=time.perf_counter()-t0
        outputs={m:[None]*len(imgs) for m in METHODS}
        for i,m,start,n,spans in mapping:
            cur=logits[start:start+n]
            if n==1:
                pred=LT.normalize_ltb(LT.greedy_ctc(cur[0],self.chars))
            else:
                events=[V3.ctc_events(x,self.chars,a,b) for x,(a,b) in zip(cur,spans)]
                pred=V3.stitch_evidence(events,spans,use_confidence=False,geometric_fallback=True)
            outputs[m][i]={'prediction':pred,'num_crops':n,'ctc_Ts':[len(x) for x in cur],
                           'spans':spans}
        return outputs, {'forward_s':inference_s,'tensors':len(inputs)}


def run_cpu_tests():
    from PIL import Image
    import torch
    class Dummy(torch.nn.Module):
        def forward(self,x):
            b,_,h,w=x.shape
            t=w//4
            out=torch.zeros(b,t,3,device=x.device)
            out[...,0]=1.0
            out[...,1]=-0.1
            out[:,t//3,1]=2
            return out
    img=Image.new('RGB',(120,30),'white')
    assert [int(x.shape[-1]) for x in [resized_tensor(img,1),resized_tensor(img,2),resized_tensor(img,3)]]==[128,256,384]
    assert len(forced_crops(img,2))==2 and len(forced_crops(img,3))==3
    assert len(forced_crops(img,3)[0][0].getdata())>0
    p=Probe(Dummy(),['blank','a','b'],'cpu',batch_size=4)
    result,perf=p.batch([img,Image.new('RGB',(210,30),'white')])
    assert set(result)==set(METHODS) and perf['tensors']==2*11
    assert all(len(v)==2 and isinstance(v[0]['prediction'],str) for v in result.values())
    assert result['baseline'][0]['ctc_Ts']==[32]
    assert result['width_x3'][0]['ctc_Ts']==[96]
    assert result['force_3'][0]['num_crops']==3
    return {'passed':True,'methods':list(METHODS),'total_tensors_for_2_imgs':perf['tensors']}

In [ ]:
%%writefile unified_router.py
"""A fully source-image/CTC-driven router; historical IDs/labels are never input."""
from __future__ import annotations
from PIL import Image
import time
import torch
import longtext_final as LT
import longtext_v3 as V3
import orientation_probe as OP
import density_probe as DP

class UnifiedRecognizer:
    def __init__(self,net,chars,device='cuda',batch_size=2):
        self.base=V3.EvidenceLongTextRecognizer(net,chars,device=device,batch_size=batch_size)
        self.orientation=OP.OrientationProbe(net,chars,device=device,batch_size=batch_size)
        self.dense=DP.Probe(net,chars,device=device,batch_size=batch_size)
        self.chars=list(chars)

    @staticmethod
    def classify(w,h,n):
        # Original raw-pixel geometry ONLY. ID, filename, GT and reference prediction unused.
        if int(w)<=int(h):
            assert int(n)==1, ('Tall expected a single original chunk',w,h,n)
            return 'orientation'
        if int(n)==1:
            return 'dense'
        return 'v3'

    def dense3_x3(self,images):
        # Reproduces density_probe.Probe.batch(...)["force_3_width_x3"] EXACTLY,
        # without paying for the five unused ablation branches.
        tensors=[];meta=[]
        for im in images:
            parts=DP.forced_crops(im,3)
            start=len(tensors)
            tensors.extend(DP.resized_tensor(p,3) for p,_,_ in parts)
            spans=[(a,b) for _,a,b in parts]
            meta.append((start,len(parts),spans))
        logits=self.dense._forward_tensors(tensors)
        ret=[]
        for start,n,spans in meta:
            cur=logits[start:start+n]
            if n==1:
                pred=LT.normalize_ltb(LT.greedy_ctc(cur[0],self.chars))
            else:
                events=[V3.ctc_events(x,self.chars,a,b) for x,(a,b) in zip(cur,spans)]
                pred=V3.stitch_evidence(events,spans,use_confidence=False,geometric_fallback=True)
            ret.append({'pred':pred,'dense_n_crops':n,'dense_ctc_Ts':[len(x) for x in cur]})
        return ret,{'dense_extra_tensors':len(tensors)}

    @torch.inference_mode()
    def infer_batch(self,images):
        images=[im.convert('RGB') for im in images]
        t0=time.perf_counter()
        # Independent matched baseline on EVERY image; the same official v3 Geometry.
        baseline=self.base.predict_many(images,method='v3_geom',return_info=True)
        routings=[self.classify(im.width,im.height,b['num_windows']) for im,b in zip(images,baseline)]
        rows=[{'route':route,'baseline_pred':b['prediction'],'baseline_num_windows':b['num_windows'],
               'pred':b['prediction'],'angle_choice':'', 'dense_n_crops':0,
               'orientation_agreement_delta':None,'orientation_conf_delta':None}
              for route,b in zip(routings,baseline)]
        ori_idx=[i for i,r in enumerate(routings) if r=='orientation']
        if ori_idx:
            ori,ori_perf=self.orientation.infer_batch([images[i] for i in ori_idx])
            for i,v in zip(ori_idx,ori):
                # Assert two independent original-orientation implementations agree.
                assert v['original']==rows[i]['baseline_pred'], ('Tall original parity',i)
                rows[i]['pred']=v['select_overlap']
                rows[i]['angle_choice']=v['overlap_choice']
                rows[i]['orientation_agreement_delta']=float(v['cw_agreement']-v['ccw_agreement'])
                rows[i]['orientation_conf_delta']=float(v['cw_confidence']-v['ccw_confidence'])
        else:ori_perf={'num_tensors':0}
        dense_idx=[i for i,r in enumerate(routings) if r=='dense']
        if dense_idx:
            preds,dense_perf=self.dense3_x3([images[i] for i in dense_idx])
            for i,v in zip(dense_idx,preds):
                rows[i]['pred']=v['pred']
                rows[i]['dense_n_crops']=int(v['dense_n_crops'])
                rows[i]['dense_ctc_Ts']=v['dense_ctc_Ts']
        else:dense_perf={'dense_extra_tensors':0}
        for r in rows:
            r.setdefault('dense_ctc_Ts',[])
        return rows,{'elapsed_seconds':time.perf_counter()-t0,
                     'baseline_images':len(images),'orientation_images':len(ori_idx),'dense_images':len(dense_idx),
                     'orientation_extra_tensors':ori_perf['num_tensors'],
                     'dense_extra_tensors':dense_perf['dense_extra_tensors']}

def run_cpu_tests():
    class Dummy(torch.nn.Module):
        def forward(self,x):
            b,_,h,w=x.shape
            t=w//4
            logits=torch.zeros((b,t,3),device=x.device)
            logits[:,:,0]=2.0
            logits[:,max(1,t//4),1]=4.0
            return logits
    net=Dummy()
    p=UnifiedRecognizer(net,['blank','a','b'],device='cpu',batch_size=2)
    imgs=[Image.new('RGB',(18,210),'white'),Image.new('RGB',(120,30),'white'),Image.new('RGB',(330,20),'white')]
    pred,perf=p.infer_batch(imgs)
    assert [x['route'] for x in pred]==['orientation','dense','v3'], pred
    assert pred[0]['angle_choice'] in ('cw','ccw')
    assert pred[1]['dense_n_crops']==3
    assert pred[2]['baseline_num_windows']>=2
    assert all(isinstance(x['pred'],str) for x in pred)
    # Dense-only wrapper must decode IDENTICALLY to the pre-existing full probe.
    all_dense,_=p.dense.batch([imgs[1]])
    only,_=p.dense3_x3([imgs[1]])
    assert only[0]['pred']==all_dense['force_3_width_x3'][0]['prediction']
    assert only[0]['dense_ctc_Ts']==all_dense['force_3_width_x3'][0]['ctc_Ts']
    return {'passed':True,'routes':[x['route'] for x in pred],'dense_parity':True}

In [ ]:
import importlib
sys.path.insert(0,str(WORK))
import longtext_final as LT,longtext_v3 as V3,orientation_probe as OP,density_probe as DP,unified_router as UNI
for m in (LT,V3,OP,DP,UNI):importlib.reload(m)
print('LT regression:',LT.run_regression_tests())
print('V3 regression:',V3.run_regression_tests_v3()['passed'])
print('Orientation regression:',OP.run_cpu_tests())
print('Dense regression:',DP.run_cpu_tests())
print('Unified v1 regression:',UNI.run_cpu_tests())

In [ ]:
from pathlib import Path
Path('sage_str_v2').mkdir(exist_ok=True)
Path('sage_str_v2/__init__.py').write_text("from .selector import Hypothesis, Observation, SelectiveGate, feature_dict\n__all__ = ['Hypothesis','Observation','SelectiveGate','feature_dict','from_logits']\nfrom .ctc_evidence import from_logits\n",encoding='utf-8')
print('Wrote sage_str_v2/__init__.py')

In [ ]:
from pathlib import Path
Path('sage_str_v3').mkdir(exist_ok=True)
Path('sage_str_v3/__init__.py').write_text('')
Path('sage_str_v2/__init__.py').write_text('from .selector import Hypothesis, Observation\n')


In [ ]:
%%writefile sage_str_v2/selector.py
"""Typed candidate record contract shared with frozen SVTRv2-T GPU adapter.
The former SAGE v2 binary gate is intentionally NOT used here.
"""
from __future__ import annotations
from dataclasses import dataclass, field
from difflib import SequenceMatcher
from math import log1p, isfinite
from typing import Iterable, Optional, Sequence
import json
import numpy as np
from scipy.stats import beta
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier

STRATEGIES = {
    "rotate_cw", "rotate_ccw", "full_width2", "geometry_v3",
    "crop2_msr", "crop3_msr", "crop3_width2", "crop3_width3",
}
NUMERIC_FEATURES = [
    "aspect_log", "base_nchars", "base_ctc_pressure", "base_entropy",
    "base_char_margin", "base_edge_occupancy", "base_confidence",
    "cand_nchars", "cand_ctc_pressure", "cand_entropy",
    "cand_char_margin", "cand_edge_occupancy", "cand_confidence",
    "prediction_agreement", "char_growth", "conf_delta", "pressure_delta",
    "candidate_overlap_agreement", "candidate_n_windows",
    "candidate_log_distortion", "candidate_cost_ms",
]
CATEGORICAL_FEATURES = ["strategy"]


def _normalize(s: str) -> str:
    return "".join(ch for ch in s.lower() if ch.isascii() and ch.isalnum())


def _repeats(s: str) -> int:
    return sum(s[i] == s[i-1] for i in range(1,len(s)))


@dataclass(frozen=True)
class Hypothesis:
    """GT-blind recognizer output and evidence from CTC/raw geometry.

    Expected confidence/entropy/margin are post-softmax CTC aggregates, not
    necessarily comparable across input transforms without calibration.
    distortion = max((W'/H')/(W/H), reciprocal); >=1.
    """
    strategy: str
    text: str
    ctc_T: int
    confidence: float = 0.0
    entropy: float = 0.0
    char_margin: float = 0.0
    edge_occupancy: float = 0.0
    overlap_agreement: float = 0.0
    n_windows: int = 1
    distortion: float = 1.0
    cost_ms: float = 0.0

    def __post_init__(self):
        if not isinstance(self.text,str):raise ValueError("text must be string")
        if self.ctc_T<1 or self.n_windows<1 or self.distortion<1:
            raise ValueError("Invalid T/windows/distortion")
        for k in ("confidence","entropy","char_margin","edge_occupancy",
                  "overlap_agreement","distortion","cost_ms"):
            if not isfinite(float(getattr(self,k))):raise ValueError(k)

    @property
    def normalized(self)->str:return _normalize(self.text)

    @property
    def pressure(self)->float:
        # Necessary CTC slots, NOT a reliable estimate of actual GT length.
        s=self.normalized
        return (len(s)+_repeats(s))/max(1,self.ctc_T)


@dataclass(frozen=True)
class Observation:
    """Inference interface NEVER has ID, filename, ground-truth or dataset."""
    width: int
    height: int
    original: Hypothesis
    alternatives: tuple[Hypothesis,...] = field(default_factory=tuple)

    def __post_init__(self):
        if self.width<=0 or self.height<=0:raise ValueError("Invalid image shape")
        if self.original.strategy!="original":raise ValueError("baseline must be original")
        names=[v.strategy for v in self.alternatives]
        if len(names)!=len(set(names)) or not set(names).issubset(STRATEGIES):
            raise ValueError("Duplicate/unknown candidate strategy")




In [ ]:
from pathlib import Path
Path('sage_str_v2').mkdir(exist_ok=True)
Path('sage_str_v2/ctc_evidence.py').write_text('"""Frozen SVTRv2-T CTC-logit evidence for SAGE-STR v2.\n\nThe class 0 is blank; logits are shaped [T,C]. No GT/label input is accepted.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Sequence\nimport numpy as np\nfrom .selector import Hypothesis\n\n\ndef _softmax(x):\n    x=x-np.max(x,axis=-1,keepdims=True)\n    e=np.exp(x)\n    return e/np.sum(e,axis=-1,keepdims=True)\n\n\ndef from_logits(logits:np.ndarray,chars:Sequence[str],strategy:str,\n                n_windows:int=1,overlap_agreement:float=0.,\n                distortion:float=1.,cost_ms:float=0.) -> Hypothesis:\n    """Compute a GT-free evidence summary from a *single* CTC sequence.\n\n    For a tiled proposal, compute one summary per crop and independently\n    aggregate events in the GPU adapter; stitched-text evidence cannot be\n    obtained by simply concatenating logits from overlapping windows.\n    """\n    z=np.asarray(logits,dtype=np.float64)\n    if z.ndim!=2 or len(z)==0 or len(z[0])!=len(chars):\n        raise ValueError(\'Expect non-empty CTC logits [T, len(chars)]\')\n    p=_softmax(z)\n    ids=np.argmax(p,axis=1)\n    if len(chars)<2:raise ValueError(\'CTC charset needs blank and characters\')\n    tokens=[];event_steps=[];event_confs=[];event_margins=[]\n    prev=0\n    for t,k in enumerate(ids):\n        k=int(k)\n        if k!=0 and k!=prev:\n            tokens.append(chars[k]);event_steps.append(t)\n            event_confs.append(float(p[t,k]))\n            sorted2=np.partition(p[t],-2)[-2:]\n            event_margins.append(float(sorted2.max()-sorted2.min()))\n        prev=k\n    entropy=-(p*np.log(np.clip(p,1e-12,1))).sum(axis=1)\n    mean_entropy=float(np.mean(entropy/np.log(len(chars))))\n    if event_steps:\n        edge=float(np.mean([(t<.1*len(z) or t>=.9*len(z)) for t in event_steps]))\n    else:edge=0.\n    return Hypothesis(strategy=strategy,text=\'\'.join(tokens),ctc_T=len(z),\n                      confidence=float(np.mean(event_confs)) if event_confs else 0.,\n                      entropy=mean_entropy,\n                      char_margin=float(np.mean(event_margins)) if event_margins else 0.,\n                      edge_occupancy=edge,overlap_agreement=overlap_agreement,\n                      n_windows=n_windows,distortion=distortion,cost_ms=cost_ms)\n',encoding='utf-8')
print('Wrote sage_str_v2/ctc_evidence.py')

In [ ]:
%%writefile sage_str_v3/selector.py
"""SAGE-STR v3 — deterministic dual-arbiter experiment, NOT validated superiority.

Frozen SVTRv2-T proposals are represented by the immutable v2 Hypothesis/Observation.
Training and tuning can see labels, but select() only receives an Observation.
No source ID, filenames, GT, benchmark labels, wall clock, or GPU latency feature at inference.
"""
from __future__ import annotations
from dataclasses import dataclass
from difflib import SequenceMatcher
from math import log1p
import json
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sage_str_v2.selector import Hypothesis, Observation
from rapidfuzz.distance.Levenshtein import distance as edit_distance

SEED = 20261118
NEUTRAL, GAIN, HARM = 0, 1, 2
METHOD_COST_UNITS = {"rotate_cw":1., "rotate_ccw":1.,"full_width2":.7,
                     "crop2_msr":1.4,"crop3_msr":1.8,"crop3_width2":2.0,
                     "crop3_width3":2.3,"geometry_v3":1.5,"original":0.}
NUMERIC = ["aspect_log","original_len","original_slot_pressure","original_entropy",
 "original_edge_occupancy","reference_len","reference_is_geometry",
 "candidate_len","relative_len_change","candidate_n_windows", "candidate_log_distortion",
 "candidate_overlap_agreement", "candidate_entropy", "candidate_confidence",
 "candidate_char_margin","sequence_agreement","candidate_longer_than_ref",
 "length_ratio", "reference_confidence", "reference_char_margin"]
CATEGORICAL = ["strategy"]
COLUMNS = CATEGORICAL + NUMERIC


def _norm(s):
    return ''.join(ch for ch in s.lower() if ch.isascii() and ch.isalnum())


def reference(obs:Observation)->Hypothesis:
    # Geometry is a safer long-line reference than single full-image MSR when
    # its prediction differs. If it equals original, same text is retained.
    return next((c for c in obs.alternatives if c.strategy=="geometry_v3"),obs.original)


def candidates(obs:Observation):
    ref=reference(obs)
    cs=[c for c in obs.alternatives if c.strategy!="geometry_v3" and c.normalized!=ref.normalized]
    if ref.strategy=="geometry_v3" and obs.original.normalized!=ref.normalized:
        cs.append(obs.original)  # Original remains a *possible*, not mandatory fallback.
    cs=sorted(cs,key=lambda c:(c.strategy,c.normalized))
    assert len({c.strategy for c in cs})==len(cs)
    return ref,cs


def group_for(c):
    return "orientation" if c.strategy in ("rotate_cw","rotate_ccw") else "recovery"


def features(obs:Observation,ref:Hypothesis,c:Hypothesis):
    a,b=ref.normalized,c.normalized
    return {
        "strategy":c.strategy,
        "aspect_log":float(np.log(obs.width/obs.height)),
        "original_len":float(len(obs.original.normalized)),
        "original_slot_pressure":float(min(2.,obs.original.pressure)),
        "original_entropy":float(obs.original.entropy),
        "original_edge_occupancy":float(obs.original.edge_occupancy),
        "reference_len":float(len(a)),
        "reference_is_geometry":float(ref.strategy=="geometry_v3"),
        "candidate_len":float(len(b)),
        "relative_len_change":float(len(b)-len(a)),
        "candidate_n_windows":float(c.n_windows),
        "candidate_log_distortion":float(log1p(c.distortion-1)),
        "candidate_overlap_agreement":float(c.overlap_agreement),
        "candidate_entropy":float(c.entropy),
        "candidate_confidence":float(c.confidence),
        "candidate_char_margin":float(c.char_margin),
        "sequence_agreement":float(SequenceMatcher(None,a,b,autojunk=False).ratio()),
        "candidate_longer_than_ref":float(len(b)>len(a)),
        "length_ratio":float((len(b)+1)/(len(a)+1)),
        "reference_confidence":float(ref.confidence),
        "reference_char_margin":float(ref.char_margin),
    }


def matrix(xs):
    z=pd.DataFrame(xs,columns=COLUMNS)
    if len(z)==0:return z
    assert np.isfinite(z[NUMERIC].to_numpy(dtype=float)).all()
    return z


def _pre():
    return ColumnTransformer([
        ("num",StandardScaler(),NUMERIC),
        ("cat",OneHotEncoder(handle_unknown="ignore"),CATEGORICAL),
    ])


def _model_clf(X,y,sample_weight):
    if len(set(y))==1:
        return DummyClassifier(strategy="constant",constant=y[0]).fit(X,y,sample_weight=sample_weight)
    m=Pipeline([("pre",_pre()),("clf",LogisticRegression(C=.4,solver="lbfgs",max_iter=1400,random_state=SEED))])
    m.fit(X,y,clf__sample_weight=sample_weight)
    return m


def _model_reg(X,y,sample_weight):
    if len(set(float(round(z,8)) for z in y))==1:
        return DummyRegressor(strategy="constant",constant=float(y[0])).fit(X,y,sample_weight=sample_weight)
    m=Pipeline([("pre",_pre()),("reg",Ridge(alpha=8.0))])
    m.fit(X,y,reg__sample_weight=sample_weight)
    return m


def _triples(records):
    """Input (Observation,GT,source) triples; source is for reporting ONLY."""
    pools={"orientation":[],"recovery":[]}
    for obs,gt,source in records:
        truth=_norm(gt)
        if not truth:continue
        ref,cs=candidates(obs)
        ref_ok=(ref.normalized==truth)
        ref_edit=edit_distance(ref.normalized,truth)
        k=max(1,len(cs))
        for c in cs:
            if c.normalized==ref.normalized:continue
            ok=(c.normalized==truth)
            label=GAIN if ok and not ref_ok else HARM if ref_ok and not ok else NEUTRAL
            cer_delta=(ref_edit-edit_distance(c.normalized,truth))/max(1,len(truth))
            pools[group_for(c)].append((features(obs,ref,c),label,float(np.clip(cer_delta,-1.,1.)),1./k))
    return pools


def _temp_prob(model,X,temp:float):
    p=model.predict_proba(X)
    out=np.zeros((len(X),3),dtype=float)
    for j,cls in enumerate(model.classes_):out[:,int(cls)]=p[:,j]
    if temp!=1. and len(model.classes_)>1:
        rows=out>0
        logp=np.where(rows,np.log(np.clip(out,1.e-12,1.)), -1.e6)
        logp=logp/temp
        logp=logp-np.max(logp,axis=1,keepdims=True)
        ex=np.exp(logp);out=ex/np.sum(ex,axis=1,keepdims=True)
    assert np.allclose(out.sum(axis=1),1.,atol=1.e-9)
    return out


@dataclass
class _Head:
    classifier:object
    regressor:object
    temperature:float=1.
    count:int=0


class DualArbiter:
    """Three-class separate arbiter heads, CER-aware reward, locked tuning.

    NO guarantee of risk control on OOD data. Audit is a diagnostic gate only.
    """
    def __init__(self,harm_weight=2.5,cer_weight=1.0,cost_weight=.002):
        self.harm_weight=float(harm_weight)
        self.cer_weight=float(cer_weight)
        self.cost_weight=float(cost_weight)
        self.heads={};self.thresholds={"orientation":float("inf"),"recovery":float("inf")}
        self.audited=False;self.audit_report={};self.tuning_report={}

    def fit(self,train):
        pools=_triples(train)
        self.heads={}
        for group,rows in pools.items():
            if not rows:continue
            X=matrix([v[0] for v in rows]); y=np.array([v[1] for v in rows],dtype=int)
            ce=np.array([v[2] for v in rows],dtype=float)
            weights=np.array([v[3] for v in rows],dtype=float)
            clf=_model_clf(X,y,weights)
            reg=_model_reg(X,ce,weights)
            self.heads[group]=_Head(clf,reg,1.,len(rows))
        if not self.heads:raise RuntimeError("No training candidates")
        self.audited=False
        return self

    def calibrate(self,records):
        pools=_triples(records);report={}
        for group,head in self.heads.items():
            rows=pools[group]
            if len(rows)<20 or len(head.classifier.classes_)<2:
                report[group]={"temperature":1.,"n":len(rows),"reason":"insufficient_data"};continue
            X=matrix([v[0] for v in rows]);targets=np.array([v[1] for v in rows],dtype=int)
            def nll(logt):
                prob=_temp_prob(head.classifier,X,float(np.exp(logt)))
                return -float(np.mean(np.log(np.clip(prob[np.arange(len(targets)),targets],1.e-12,1.))))
            result=minimize_scalar(nll,bounds=(-1.4,1.4),method="bounded",options={"xatol":1.e-5})
            head.temperature=float(np.exp(result.x))
            report[group]={"temperature":head.temperature,"n":len(rows),"nll":float(result.fun)}
        return report

    def _rank(self,obs):
        ref,cs=candidates(obs)
        ranked=[]
        for group in ("orientation","recovery"):
            use=[c for c in cs if group_for(c)==group]
            if not use or group not in self.heads:continue
            head=self.heads[group];X=matrix([features(obs,ref,c) for c in use])
            p=_temp_prob(head.classifier,X,head.temperature)
            cer=np.clip(head.regressor.predict(X),-.75,.75)
            for c,pr,ce in zip(use,p,cer):
                fixed_cost=self.cost_weight*METHOD_COST_UNITS.get(c.strategy,2.)
                utility=float(pr[GAIN]-self.harm_weight*pr[HARM]+self.cer_weight*ce-fixed_cost)
                ranked.append({"hyp":c,"group":group,"strategy":c.strategy,
                               "reference":ref.strategy,"score":utility,
                               "p_gain":float(pr[GAIN]),"p_harm":float(pr[HARM]),
                               "p_neither":float(pr[NEUTRAL]),
                               "predicted_cer_gain":float(ce),
                               "fixed_cost":float(fixed_cost)})
        ranked.sort(key=lambda r:(-r["score"],r["strategy"]))
        return ref,ranked

    @staticmethod
    def _decision_from_ranked(ref,ranked,thresholds):
        eligible=[x for x in ranked if x["score"]>thresholds[x["group"]]]
        return eligible[0]["hyp"] if eligible else ref

    def propose(self,obs,thresholds=None):
        ref,ranked=self._rank(obs)
        return self._decision_from_ranked(ref,ranked,self.thresholds if thresholds is None else thresholds)

    def select(self,obs):
        ref,ranked=self._rank(obs)
        if not self.audited:
            chosen=ref;reason="audit_failed_reference_fallback"
        else:
            chosen=self._decision_from_ranked(ref,ranked,self.thresholds)
            reason="accepted_evidence" if chosen is not ref else "reference_fallback"
        return {"prediction":chosen.normalized,"chosen":chosen.strategy,
                "reference":ref.strategy,"reason":reason}

    def rank_trace(self,obs):
        ref,r=self._rank(obs)
        selected=self.select(obs)
        return [{k:v for k,v in d.items() if k!='hyp'}|{
            "prediction":d['hyp'].normalized,"accepted":d['strategy']==selected['chosen'],
            "threshold":self.thresholds[d['group']]}
            for d in r]

    def _evaluate_thresholds(self,precomputed,thresholds):
        aggregate={"wins":0,"harms":0,"edit_gain":0,"promoted":0,"n":0}
        by_source={}
        for r in precomputed:
            ref,ranked,gt,src=r
            d=self._decision_from_ranked(ref,ranked,thresholds)
            winner=int(d.normalized==gt and ref.normalized!=gt)
            harm=int(d.normalized!=gt and ref.normalized==gt)
            edit_gain=edit_distance(ref.normalized,gt)-edit_distance(d.normalized,gt)
            for a in (aggregate,by_source.setdefault(src,{"wins":0,"harms":0,"edit_gain":0,"promoted":0,"n":0})):
                a['wins']+=winner;a['harms']+=harm;a['edit_gain']+=edit_gain
                a['promoted']+=int(d.normalized!=ref.normalized);a['n']+=1
        return aggregate,by_source

    def _precompute(self,records):
        return [(self._rank(o)[0],self._rank(o)[1],_norm(gt),src)
                for o,gt,src in records if _norm(gt)]

    def choose_thresholds(self,records):
        """Tuning split ONLY; score predeclared and risk is descriptive."""
        if not records:raise ValueError("Empty policy tune")
        rows=self._precompute(records)
        GRID=(-.20,-.10,0.,.10,.20,.40,.65,float('inf'))
        best=None
        for t_ori in GRID:
            for t_rec in GRID:
                ts={"orientation":t_ori,"recovery":t_rec}
                agg,src=self._evaluate_thresholds(rows,ts)
                if agg['harms']/max(1,agg['n'])>.025:continue
                if any(v['n']>=35 and v['harms']-v['wins']>2 for v in src.values()):continue
                objective=agg['wins']-2.5*agg['harms']+0.03*agg['edit_gain']
                key=(objective,-agg['harms'],-agg['promoted'])
                if best is None or key>best[0]:best=(key,ts,agg,src)
        self.thresholds=best[1] if best else {"orientation":float('inf'),"recovery":float('inf')}
        self.audited=False
        self.tuning_report={"thresholds":self._printable_thresholds(),
                            "tune":best[2] if best else {},"by_source":best[3] if best else {}}
        return self.tuning_report

    def audit(self,records,min_n=100):
        if not records:raise ValueError('Empty audit')
        rows=self._precompute(records)
        agg,by_source=self._evaluate_thresholds(rows,self.thresholds)
        # This is an empirical audit guard, NOT a guaranteed bound under shift.
        passed=bool(agg['n']>=min_n and agg['wins']>=agg['harms'] and
            agg['harms']/agg['n']<=.025 and
            all(v['wins']+1>=v['harms'] for v in by_source.values() if v['n']>=30))
        self.audited=passed
        self.audit_report={"passed":passed,"n":agg['n'],"summary":agg,"by_source":by_source,
                           "note":"Development audit; no OOD guarantee or conformal claim"}
        return self.audit_report

    def _printable_thresholds(self):
        return {k:(v if np.isfinite(v) else None) for k,v in self.thresholds.items()}

    def manifest(self):
        return {"version":"SAGE-STR-v3-experimental","seed":SEED,
           "thresholds":self._printable_thresholds(),"audit":self.audit_report,
           "calibration_temps":{k:v.temperature for k,v in self.heads.items()},
           "features":COLUMNS,"cost":"fixed token units, not GPU timings",
           "training_classes":"mutually exclusive neutral/gain/harm",
           "reference":"geometry_v3 when available, otherwise original",
           "risk_note":"No guarantee on unseen distributions"}


In [ ]:
from pathlib import Path
Path("sage_str_v4").mkdir(parents=True,exist_ok=True)
Path("sage_str_v4/__init__.py").write_text("")


In [ ]:
%%writefile sage_str_v4/selector.py
"""SAGE-STR v4 — contextual dual-arbiter experimental selector, NOT validated superiority.

Frozen SVTRv2-T proposals are represented by the immutable v2 Hypothesis/Observation.
Training and tuning can see labels, but select() only receives an Observation.
No source ID, filenames, GT, benchmark labels, wall clock, or GPU latency feature at inference.
"""
from __future__ import annotations
from dataclasses import dataclass
from difflib import SequenceMatcher
from math import log1p
import json
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sage_str_v2.selector import Hypothesis, Observation
from rapidfuzz.distance.Levenshtein import distance as edit_distance

SEED = 20261118
NEUTRAL, GAIN, HARM = 0, 1, 2
METHOD_COST_UNITS = {"rotate_cw":1., "rotate_ccw":1.,"full_width2":.7,
                     "crop2_msr":1.4,"crop3_msr":1.8,"crop3_width2":2.0,
                     "crop3_width3":2.3,"geometry_v3":1.5,"original":0.}
NUMERIC = ["aspect_log","original_len","original_slot_pressure","original_entropy",
 "original_edge_occupancy","reference_len","reference_is_geometry",
 "candidate_len","relative_len_change","candidate_n_windows", "candidate_log_distortion",
 "candidate_overlap_agreement", "candidate_entropy", "candidate_confidence",
 "candidate_char_margin","sequence_agreement","candidate_longer_than_ref",
 "length_ratio", "reference_confidence", "reference_char_margin"]
NUMERIC += ['confidence_diff_to_reference', 'entropy_diff_to_reference', 'margin_diff_to_reference', 'confidence_diff_to_original', 'ref_original_agreement', 'reference_len_minus_original_len', 'reference_vs_original_length_ratio', 'candidate_len_minus_original_len', 'candidate_original_agreement', 'rival_orientation_available', 'rival_orientation_agreement', 'rival_orientation_conf_gap', 'rival_orientation_len_gap', 'rotation_confidence_rank', 'reference_is_truncated_vs_candidate', 'n_available_alternatives']
CATEGORICAL = ["strategy"]
COLUMNS = CATEGORICAL + NUMERIC


def _norm(s):
    return ''.join(ch for ch in s.lower() if ch.isascii() and ch.isalnum())


def reference(obs:Observation)->Hypothesis:
    # Geometry is a safer long-line reference than single full-image MSR when
    # its prediction differs. If it equals original, same text is retained.
    return next((c for c in obs.alternatives if c.strategy=="geometry_v3"),obs.original)


def candidates(obs:Observation):
    ref=reference(obs)
    cs=[c for c in obs.alternatives if c.strategy!="geometry_v3" and c.normalized!=ref.normalized]
    if ref.strategy=="geometry_v3" and obs.original.normalized!=ref.normalized:
        cs.append(obs.original)  # Original remains a *possible*, not mandatory fallback.
    cs=sorted(cs,key=lambda c:(c.strategy,c.normalized))
    assert len({c.strategy for c in cs})==len(cs)
    return ref,cs


def group_for(c):
    return "orientation" if c.strategy in ("rotate_cw","rotate_ccw") else "recovery"


def features(obs:Observation,ref:Hypothesis,c:Hypothesis):
    a,b=ref.normalized,c.normalized
    base = {
        "strategy":c.strategy,
        "aspect_log":float(np.log(obs.width/obs.height)),
        "original_len":float(len(obs.original.normalized)),
        "original_slot_pressure":float(min(2.,obs.original.pressure)),
        "original_entropy":float(obs.original.entropy),
        "original_edge_occupancy":float(obs.original.edge_occupancy),
        "reference_len":float(len(a)),
        "reference_is_geometry":float(ref.strategy=="geometry_v3"),
        "candidate_len":float(len(b)),
        "relative_len_change":float(len(b)-len(a)),
        "candidate_n_windows":float(c.n_windows),
        "candidate_log_distortion":float(log1p(c.distortion-1)),
        "candidate_overlap_agreement":float(c.overlap_agreement),
        "candidate_entropy":float(c.entropy),
        "candidate_confidence":float(c.confidence),
        "candidate_char_margin":float(c.char_margin),
        "sequence_agreement":float(SequenceMatcher(None,a,b,autojunk=False).ratio()),
        "candidate_longer_than_ref":float(len(b)>len(a)),
        "length_ratio":float((len(b)+1)/(len(a)+1)),
        "reference_confidence":float(ref.confidence),
        "reference_char_margin":float(ref.char_margin),
    }


    original=obs.original
    rot=[a for a in obs.alternatives if a.strategy in ("rotate_cw","rotate_ccw")]
    rival=next((a for a in rot if a.strategy!=c.strategy),None) if c.strategy.startswith('rotate_') else None
    # Candidate-set context: both rotations are observed for tall images, allowing
    # discrimination between actual reorientation and short-word hallucination.
    rotate_rank=float(sum(a.confidence>c.confidence for a in rot)) if c.strategy.startswith('rotate_') else 0.
    base.update({
        "confidence_diff_to_reference":float(c.confidence-ref.confidence),
        "entropy_diff_to_reference":float(c.entropy-ref.entropy),
        "margin_diff_to_reference":float(c.char_margin-ref.char_margin),
        "confidence_diff_to_original":float(c.confidence-original.confidence),
        "ref_original_agreement":float(SequenceMatcher(None,a,original.normalized,autojunk=False).ratio()),
        "reference_len_minus_original_len":float(len(a)-len(original.normalized)),
        "reference_vs_original_length_ratio":float((len(a)+1.)/(len(original.normalized)+1.)),
        "candidate_len_minus_original_len":float(len(b)-len(original.normalized)),
        "candidate_original_agreement":float(SequenceMatcher(None,b,original.normalized,autojunk=False).ratio()),
        "rival_orientation_available":float(rival is not None),
        "rival_orientation_agreement":float(SequenceMatcher(None,b,rival.normalized,autojunk=False).ratio()) if rival else 0.,
        "rival_orientation_conf_gap":float(c.confidence-rival.confidence) if rival else 0.,
        "rival_orientation_len_gap":float(len(b)-len(rival.normalized)) if rival else 0.,
        "rotation_confidence_rank":rotate_rank,
        "reference_is_truncated_vs_candidate":float(max(0,len(b)-len(a))/max(1,len(b))),
        "n_available_alternatives":float(len(obs.alternatives)),
    })
    return base

def matrix(xs):
    z=pd.DataFrame(xs,columns=COLUMNS)
    if len(z)==0:return z
    assert np.isfinite(z[NUMERIC].to_numpy(dtype=float)).all()
    return z


def _pre():
    return ColumnTransformer([
        ("num",StandardScaler(),NUMERIC),
        ("cat",OneHotEncoder(handle_unknown="ignore"),CATEGORICAL),
    ])


def _model_clf(X,y,sample_weight):
    if len(set(y))==1:
        return DummyClassifier(strategy="constant",constant=y[0]).fit(X,y,sample_weight=sample_weight)
    m=Pipeline([("pre",_pre()),("clf",LogisticRegression(C=.4,solver="lbfgs",max_iter=1400,random_state=SEED))])
    m.fit(X,y,clf__sample_weight=sample_weight)
    return m


def _model_reg(X,y,sample_weight):
    if len(set(float(round(z,8)) for z in y))==1:
        return DummyRegressor(strategy="constant",constant=float(y[0])).fit(X,y,sample_weight=sample_weight)
    m=Pipeline([("pre",_pre()),("reg",Ridge(alpha=8.0))])
    m.fit(X,y,reg__sample_weight=sample_weight)
    return m


def _triples(records):
    """Input (Observation,GT,source) triples; source is for reporting ONLY."""
    pools={"orientation":[],"recovery":[]}
    for obs,gt,source in records:
        truth=_norm(gt)
        if not truth:continue
        ref,cs=candidates(obs)
        ref_ok=(ref.normalized==truth)
        ref_edit=edit_distance(ref.normalized,truth)
        k=max(1,len(cs))
        for c in cs:
            if c.normalized==ref.normalized:continue
            ok=(c.normalized==truth)
            label=GAIN if ok and not ref_ok else HARM if ref_ok and not ok else NEUTRAL
            cer_delta=(ref_edit-edit_distance(c.normalized,truth))/max(1,len(truth))
            pools[group_for(c)].append((features(obs,ref,c),label,float(np.clip(cer_delta,-1.,1.)),1./k))
    return pools


def _temp_prob(model,X,temp:float):
    p=model.predict_proba(X)
    out=np.zeros((len(X),3),dtype=float)
    for j,cls in enumerate(model.classes_):out[:,int(cls)]=p[:,j]
    if temp!=1. and len(model.classes_)>1:
        rows=out>0
        logp=np.where(rows,np.log(np.clip(out,1.e-12,1.)), -1.e6)
        logp=logp/temp
        logp=logp-np.max(logp,axis=1,keepdims=True)
        ex=np.exp(logp);out=ex/np.sum(ex,axis=1,keepdims=True)
    assert np.allclose(out.sum(axis=1),1.,atol=1.e-9)
    return out


@dataclass
class _Head:
    classifier:object
    regressor:object
    temperature:float=1.
    count:int=0


class ContextualDualArbiter:
    """V4: candidate-set features, three-class calibrated heads, fixed-cost utility.

    NO guarantee of risk control on OOD data. Audit is a diagnostic gate only.
    """
    def __init__(self,harm_weight=2.5,cer_weight=1.0,cost_weight=.002):
        self.harm_weight=float(harm_weight)
        self.cer_weight=float(cer_weight)
        self.cost_weight=float(cost_weight)
        self.heads={};self.thresholds={"orientation":float("inf"),"recovery":float("inf")}
        self.audited=False;self.audit_report={};self.tuning_report={}

    def fit(self,train):
        pools=_triples(train)
        self.heads={}
        for group,rows in pools.items():
            if not rows:continue
            X=matrix([v[0] for v in rows]); y=np.array([v[1] for v in rows],dtype=int)
            ce=np.array([v[2] for v in rows],dtype=float)
            weights=np.array([v[3] for v in rows],dtype=float)
            clf=_model_clf(X,y,weights)
            reg=_model_reg(X,ce,weights)
            self.heads[group]=_Head(clf,reg,1.,len(rows))
        if not self.heads:raise RuntimeError("No training candidates")
        self.audited=False
        return self

    def calibrate(self,records):
        pools=_triples(records);report={}
        for group,head in self.heads.items():
            rows=pools[group]
            if len(rows)<20 or len(head.classifier.classes_)<2:
                report[group]={"temperature":1.,"n":len(rows),"reason":"insufficient_data"};continue
            X=matrix([v[0] for v in rows]);targets=np.array([v[1] for v in rows],dtype=int)
            def nll(logt):
                prob=_temp_prob(head.classifier,X,float(np.exp(logt)))
                return -float(np.mean(np.log(np.clip(prob[np.arange(len(targets)),targets],1.e-12,1.))))
            result=minimize_scalar(nll,bounds=(-1.4,1.4),method="bounded",options={"xatol":1.e-5})
            head.temperature=float(np.exp(result.x))
            report[group]={"temperature":head.temperature,"n":len(rows),"nll":float(result.fun)}
        return report

    def _rank(self,obs):
        ref,cs=candidates(obs)
        ranked=[]
        for group in ("orientation","recovery"):
            use=[c for c in cs if group_for(c)==group]
            if not use or group not in self.heads:continue
            head=self.heads[group];X=matrix([features(obs,ref,c) for c in use])
            p=_temp_prob(head.classifier,X,head.temperature)
            cer=np.clip(head.regressor.predict(X),-.75,.75)
            for c,pr,ce in zip(use,p,cer):
                fixed_cost=self.cost_weight*METHOD_COST_UNITS.get(c.strategy,2.)
                utility=float(pr[GAIN]-self.harm_weight*pr[HARM]+self.cer_weight*ce-fixed_cost)
                ranked.append({"hyp":c,"group":group,"strategy":c.strategy,
                               "reference":ref.strategy,"score":utility,
                               "p_gain":float(pr[GAIN]),"p_harm":float(pr[HARM]),
                               "p_neither":float(pr[NEUTRAL]),
                               "predicted_cer_gain":float(ce),
                               "fixed_cost":float(fixed_cost)})
        ranked.sort(key=lambda r:(-r["score"],r["strategy"]))
        return ref,ranked

    @staticmethod
    def _decision_from_ranked(ref,ranked,thresholds):
        eligible=[x for x in ranked if x["score"]>thresholds[x["group"]]]
        return eligible[0]["hyp"] if eligible else ref

    def propose(self,obs,thresholds=None):
        ref,ranked=self._rank(obs)
        return self._decision_from_ranked(ref,ranked,self.thresholds if thresholds is None else thresholds)

    def select(self,obs):
        ref,ranked=self._rank(obs)
        if not self.audited:
            chosen=ref;reason="audit_failed_reference_fallback"
        else:
            chosen=self._decision_from_ranked(ref,ranked,self.thresholds)
            reason="accepted_evidence" if chosen is not ref else "reference_fallback"
        return {"prediction":chosen.normalized,"chosen":chosen.strategy,
                "reference":ref.strategy,"reason":reason}

    def rank_trace(self,obs):
        ref,r=self._rank(obs)
        selected=self.select(obs)
        return [{k:v for k,v in d.items() if k!='hyp'}|{
            "prediction":d['hyp'].normalized,"accepted":d['strategy']==selected['chosen'],
            "threshold":self.thresholds[d['group']]}
            for d in r]

    def _evaluate_thresholds(self,precomputed,thresholds):
        aggregate={"wins":0,"harms":0,"edit_gain":0,"promoted":0,"n":0}
        by_source={}
        for r in precomputed:
            ref,ranked,gt,src=r
            d=self._decision_from_ranked(ref,ranked,thresholds)
            winner=int(d.normalized==gt and ref.normalized!=gt)
            harm=int(d.normalized!=gt and ref.normalized==gt)
            edit_gain=edit_distance(ref.normalized,gt)-edit_distance(d.normalized,gt)
            for a in (aggregate,by_source.setdefault(src,{"wins":0,"harms":0,"edit_gain":0,"promoted":0,"n":0})):
                a['wins']+=winner;a['harms']+=harm;a['edit_gain']+=edit_gain
                a['promoted']+=int(d.normalized!=ref.normalized);a['n']+=1
        return aggregate,by_source

    def _precompute(self,records):
        return [(self._rank(o)[0],self._rank(o)[1],_norm(gt),src)
                for o,gt,src in records if _norm(gt)]

    def choose_thresholds(self,records):
        """Tuning split ONLY; score predeclared and risk is descriptive."""
        if not records:raise ValueError("Empty policy tune")
        rows=self._precompute(records)
        GRID=(-.20,-.10,0.,.10,.20,.40,.65,float('inf'))
        best=None
        for t_ori in GRID:
            for t_rec in GRID:
                ts={"orientation":t_ori,"recovery":t_rec}
                agg,src=self._evaluate_thresholds(rows,ts)
                if agg['harms']/max(1,agg['n'])>.025:continue
                if any(v['n']>=35 and v['harms']-v['wins']>2 for v in src.values()):continue
                objective=agg['wins']-2.5*agg['harms']+0.03*agg['edit_gain']-1.5*sum(max(0,v['harms']-v['wins']) for v in src.values() if v['n']>=20)
                key=(objective,-agg['harms'],-agg['promoted'])
                if best is None or key>best[0]:best=(key,ts,agg,src)
        self.thresholds=best[1] if best else {"orientation":float('inf'),"recovery":float('inf')}
        self.audited=False
        self.tuning_report={"thresholds":self._printable_thresholds(),
                            "tune":best[2] if best else {},"by_source":best[3] if best else {}}
        return self.tuning_report

    def audit(self,records,min_n=100):
        if not records:raise ValueError('Empty audit')
        rows=self._precompute(records)
        agg,by_source=self._evaluate_thresholds(rows,self.thresholds)
        # This is an empirical audit guard, NOT a guaranteed bound under shift.
        passed=bool(agg['n']>=min_n and agg['wins']>=agg['harms'] and
            agg['harms']/agg['n']<=.025 and
            all(v['wins']+1>=v['harms'] for v in by_source.values() if v['n']>=30))
        self.audited=passed
        self.audit_report={"passed":passed,"n":agg['n'],"summary":agg,"by_source":by_source,
                           "note":"Development audit; no OOD guarantee or conformal claim"}
        return self.audit_report

    def _printable_thresholds(self):
        return {k:(v if np.isfinite(v) else None) for k,v in self.thresholds.items()}

    def manifest(self):
        return {"version":"SAGE-STR-v4-contextual-experimental","seed":SEED,
           "thresholds":self._printable_thresholds(),"audit":self.audit_report,
           "calibration_temps":{k:v.temperature for k,v in self.heads.items()},
           "features":COLUMNS,"cost":"fixed token units, not GPU timings",
           "training_classes":"mutually exclusive neutral/gain/harm",
           "reference":"geometry_v3 when available, otherwise original",
           "risk_note":"No guarantee on unseen distributions"}


In [ ]:
%%writefile sage_str_v4/batch.py
"""Deterministic vectorized inference for frozen-candidate SAGE v3/v4 selectors.

No GT, source labels, image IDs or wall time enters scoring. A reference implementation
of per-image select() is checked for parity in the notebook.
"""
from collections import defaultdict
import numpy as np


def score_observations(selector, observations, module):
    """Return (decision list, per-observation GT-blind candidate-score lists)."""
    refs=[]
    ranked=[[] for _ in observations]
    grouped=defaultdict(list)
    for i,obs in enumerate(observations):
        ref,cs=module.candidates(obs)
        refs.append(ref)
        for c in cs:
            group=module.group_for(c)
            if group in selector.heads:
                grouped[group].append((i,c,module.features(obs,ref,c)))
    for group,records in grouped.items():
        head=selector.heads[group]
        X=module.matrix([r[2] for r in records])
        proba=module._temp_prob(head.classifier,X,head.temperature)
        delta=np.clip(head.regressor.predict(X),-.75,.75)
        for (idx,c,_),pr,ce in zip(records,proba,delta):
            fixed=selector.cost_weight*module.METHOD_COST_UNITS.get(c.strategy,2.)
            utility=float(pr[module.GAIN]-selector.harm_weight*pr[module.HARM]+selector.cer_weight*ce-fixed)
            ranked[idx].append({'hyp':c,'group':group,'strategy':c.strategy,
                    'reference':refs[idx].strategy,'score':utility,
                    'p_gain':float(pr[module.GAIN]),'p_harm':float(pr[module.HARM]),
                    'p_neither':float(pr[module.NEUTRAL]),
                    'predicted_cer_gain':float(ce),'fixed_cost':float(fixed)})
    decisions=[]; trace=[]
    for ref,rr in zip(refs,ranked):
        rr.sort(key=lambda d:(-d['score'],d['strategy']))
        if selector.audited:
            chosen=selector._decision_from_ranked(ref,rr,selector.thresholds)
            reason='accepted_evidence' if chosen is not ref else 'reference_fallback'
        else:
            chosen=ref;reason='audit_failed_reference_fallback'
        decisions.append({'prediction':chosen.normalized,'chosen':chosen.strategy,
                          'reference':ref.strategy,'reason':reason})
        trace.append([{k:v for k,v in d.items() if k!='hyp'} | {
            'prediction':d['hyp'].normalized,
            'accepted':d['strategy']==chosen.strategy,
            'threshold':selector.thresholds[d['group']]}
            for d in rr])
    return decisions,trace


In [ ]:
from sage_str_v2.selector import Hypothesis, Observation
from sage_str_v2.ctc_evidence import from_logits
from sage_str_v3.selector import DualArbiter, reference, candidates, group_for, features, SEED
from scipy.stats import binomtest
from rapidfuzz.distance import Levenshtein
import pandas as pd
import torch
import sklearn
print('SAGE v3 decision-layer imports PASS; sklearn:',sklearn.__version__)

from sage_str_v4.selector import ContextualDualArbiter, features as v4_features, COLUMNS as V4_COLUMNS
print("V4 context feature count:",len(V4_COLUMNS)-1)


In [ ]:
import copy

def _h(tag,t,n=1,cost=0.):
    return Hypothesis(tag,t,ctc_T=48,confidence=.7,entropy=.12,
                      char_margin=.23,edge_occupancy=.02,overlap_agreement=.5,
                      n_windows=n,distortion=1.,cost_ms=cost)

_probe=Observation(280,24,_h('original','abcd'),
    (_h('geometry_v3','abcdefgh',n=3),_h('full_width2','abcdefg',cost=1.),_h('crop3_width3','abcdefghi',n=3,cost=22.)))
assert reference(_probe).strategy=='geometry_v3'
assert 'original' in [h.strategy for h in candidates(_probe)[1]]
assert 'geometry_v3' not in [h.strategy for h in candidates(_probe)[1]]

_records=[]
for i in range(280):
    b='abcdefg' if i%3 else 'abcdfg'
    alt='abcdefg' if i%5 else 'abcefg'
    if i%2:
        opts=(_h('rotate_cw',alt,cost=i*2),_h('rotate_ccw','abcfeg',cost=i))
        obs=Observation(40,130,_h('original',b),opts)
    else:
        opts=(_h('geometry_v3','abcdefg',n=3),_h('full_width2',alt,cost=i),_h('crop3_width3','abcdegg',n=3,cost=i*3))
        obs=Observation(300,24,_h('original',b),opts)
    _records.append((obs,'abcdefg' if i%13 else 'abcdffg',('LTB' if i%2 else 'IIIT5K')))

_cpu=DualArbiter().fit(_records[:130])
_cpu.calibrate(_records[130:190])
_cpu.choose_thresholds(_records[190:240])
_cpu.audit(_records[240:280],min_n=25)
for ob,gt,src in _records[-15:]:
    out1=_cpu.select(ob)
    assert out1==_cpu.select(ob)
    for d in _cpu.rank_trace(ob):
        assert abs(d['p_gain']+d['p_harm']+d['p_neither']-1)<1.e-9
    # GPU runtime timing MUST NOT influence utility or the decision.
    ob2=Observation(ob.width,ob.height,copy.copy(ob.original),
                    tuple(__import__('dataclasses').replace(c,cost_ms=c.cost_ms+99999.) for c in ob.alternatives))
    assert _cpu.select(ob2)==out1
print('SAGE v3 CPU invariants PASS: simplex, Geometry reference, Original candidate, deterministic cost immunity.')

from sage_str_v4.selector import reference as ref4, candidates as cand4
from sage_str_v4.selector import _temp_prob, GAIN, HARM
_features=v4_features(_probe,ref4(_probe),cand4(_probe)[1][0])
assert set(_features)==set(V4_COLUMNS)
assert np.isfinite(np.array([_features[k] for k in V4_COLUMNS if k!='strategy'],dtype=float)).all()
assert v4_features(_probe,ref4(_probe),cand4(_probe)[1][0])==_features
print("SAGE v4 context features CPU sanity PASS")


In [ ]:
# 3. Fetch official T assets (unless path overrides supplied)

def fetch_gdrive(fid,out,min_bytes):
    out=Path(out);out.parent.mkdir(exist_ok=True,parents=True)
    if out.is_file() and out.stat().st_size>=min_bytes:
        return out
    if out.exists():out.unlink()
    try:gdown.download(id=fid,output=str(out),quiet=False)
    except Exception as exc:raise RuntimeError(f'Could not download official asset {fid}: {exc}') from exc
    if not out.is_file() or out.stat().st_size<min_bytes:
        raise RuntimeError(f'Google Drive download failed or incomplete ({out}); set the matching local override.')
    return out
CKPT=Path(CKPT_OVERRIDE) if CKPT_OVERRIDE else fetch_gdrive(CKPT_ID,WORK/'longstr_svtr_t'/'best.pth',5_000_000)
CFG=Path(CFG_OVERRIDE) if CFG_OVERRIDE else fetch_gdrive(CFG_ID,WORK/'longstr_svtr_t'/'config.yml',1000)
assert CKPT.is_file() and CFG.is_file()
net,chars=LT.load_official_svtr(REPO,CFG,CKPT,DEVICE)
recognizer=UNI.UnifiedRecognizer(net,chars,device=DEVICE,batch_size=IMAGE_BATCH)

def sha256(file):
    h=hashlib.sha256()
    with open(file,'rb') as f:
        for buf in iter(lambda:f.read(1<<20),b''):h.update(buf)
    return h.hexdigest()
provenance={'OpenOCR_revision':OPENOCR_REV,'checkpoint_sha256':sha256(CKPT),'config_sha256':sha256(CFG),
            'classes':len(chars),'model':'SVTRv2-T','source_stitch':'LTB.ipynb G2', 'experiment':'external matched baseline-vs-unified, frozen parameters and GT-blind inference',
            'source_preproc':'LTB.ipynb cell 12',
            'legacy_window_ratio':LT.LEGACY_CHUNK_AR,'legacy_overlap':LT.LEGACY_OVERLAP,
            'test_protocol':'Exploratory reused LTB; all candidates predeclared, no candidate selected from test for independent claim'}
(OUT/'provenance.json').write_text(json.dumps(provenance,indent=2))
print('PASS strict checkpoint + CTC logits parity; vocab size',len(chars))
full_recognizer=LT.LongTextRecognizer(net,chars,device=DEVICE,batch_size=IMAGE_BATCH)

if provenance['checkpoint_sha256']!=EXPECTED_CKPT_SHA or provenance['config_sha256']!=EXPECTED_CFG_SHA:
    raise RuntimeError('Reference official checkpoint/config hash mismatch: no matched comparison')

In [ ]:
EVAL_MAX_PER_DATASET=MAX_EXTERNAL
CHECK_EXACT_PIXEL_OVERLAP_WITH_LTB=False

In [ ]:
# Resolve the actual official dataset file paths; no guessing about local archive names.
from huggingface_hub import HfApi, hf_hub_download
import lmdb
from contextlib import closing
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
from collections import Counter
api=HfApi()
info=api.dataset_info(HF_DATA)
DATA_REVISION=info.sha
REPO_FILES=api.list_repo_files(HF_DATA,repo_type='dataset',revision=DATA_REVISION)
print('Pinned dataset repo revision:',DATA_REVISION)

def standardized(s):
    return ''.join(ch for ch in s.lower() if ch.isalnum())

def resolve_lmdb_prefix(wanted):
    exact=wanted+'/data.mdb'
    if exact in REPO_FILES: return wanted
    candidates=[f[:-len('/data.mdb')] for f in REPO_FILES if f.endswith('/data.mdb')]
    hits=[s for s in candidates if standardized(s)==standardized(wanted)]
    if len(hits)!=1:
        raise RuntimeError(f'Cannot uniquely locate {wanted!r}; matches={hits}, example paths={candidates[:30]}')
    return hits[0]

SOURCES={}
for wanted in DATASET_NAMES:
    prefix=resolve_lmdb_prefix(wanted)
    if DATASET_ROOT_OVERRIDE is not None:
        parent=Path(DATASET_ROOT_OVERRIDE)/prefix
        path=parent/'data.mdb'
        if not path.is_file(): raise FileNotFoundError(path)
    else:
        path=Path(hf_hub_download(repo_id=HF_DATA,filename=prefix+'/data.mdb',repo_type='dataset',revision=DATA_REVISION))
    SOURCES[wanted]=path
    with closing(lmdb.open(str(path.parent),readonly=True,lock=False,readahead=False,max_readers=32)) as env:
        with env.begin() as txn:
            raw=txn.get(b'num-samples')
            if not raw:raise RuntimeError(f'LMDB {wanted} missing num-samples')
            count=int(raw.decode())
            for i in (1,count):
                if txn.get(f'image-{i:09d}'.encode()) is None or txn.get(f'label-{i:09d}'.encode()) is None:
                    raise RuntimeError(f'Malformed LMDB {wanted}: missing image/label at {i}')
    expected_n={'u14m/multi_oriented':1369,'u14m/multi_words':829,'evaluation/IIIT5k':3000,'evaluation/CUTE80':288}
    if EVAL_MAX_PER_DATASET==0 and wanted in expected_n and count!=expected_n[wanted]:
        raise RuntimeError(f'{wanted}: published count {expected_n[wanted]}, downloaded {count}; stop rather than silently change evaluation protocol')
    print(f'{wanted}: {count} samples, LMDB={path}')
manifest={'data_repo':HF_DATA,'data_revision':DATA_REVISION,
          'dataset_paths':{k:str(v.relative_to(v.parents[1])) for k,v in SOURCES.items()},
          'dataset_names':DATASET_NAMES,'max_per_dataset':EVAL_MAX_PER_DATASET,
          'ltb_exact_pixel_overlap_checked':CHECK_EXACT_PIXEL_OVERLAP_WITH_LTB}
(OUT/'dataset_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))

In [ ]:
# 4. LTB auto-fetch with strict manifest validation; no data used for train.
import tarfile
if LTB_OVERRIDE:
    LTB_ROOT=Path(LTB_OVERRIDE)
else:
    archive=Path(hf_hub_download(repo_id=HF_DATA,filename='ltb.tar.xz',repo_type='dataset',revision=DATA_REVISION))
    LTB_ROOT=WORK/'longstr_ltb'
    marker=LTB_ROOT/'extract_complete.json'
    if not marker.exists():
        LTB_ROOT.mkdir(parents=True,exist_ok=True)
        with tarfile.open(archive,'r:xz') as tar:
            mm=tar.getmembers()
            for m in mm:
                target=(LTB_ROOT/m.name).resolve()
                if not (target == LTB_ROOT.resolve() or LTB_ROOT.resolve() in target.parents):
                    raise RuntimeError(f'Unsafe archive path {m.name}')
                if not (m.isfile() or m.isdir()):
                    raise RuntimeError(f'Unsupported archive member {m.name}')
            try:tar.extractall(LTB_ROOT,members=mm,filter='data')
            except TypeError:tar.extractall(LTB_ROOT,members=mm)
        marker.write_text(json.dumps({'archive':str(archive),'n':len(mm)}))
manifest=list(LTB_ROOT.rglob('ultra_long_25_list.txt'))
if len(manifest)!=1:raise RuntimeError(f'Expected one ultra_long_25_list.txt, got {manifest}')
labels=manifest[0].read_text(encoding='utf-8-sig').splitlines()
items=[]
for i,line in enumerate(labels):
    if not line.strip():continue
    fields=line.split('\t',1)
    if len(fields)!=2:raise RuntimeError(f'Manifest bad row: {line[:100]!r}')
    rel,gt=fields
    rel=rel.strip().strip('"');q=Path(rel)
    candidates=[manifest[0].parent/q,LTB_ROOT/q,manifest[0].parent.parent/q,
                manifest[0].parent/'img'/q.name,LTB_ROOT/'img'/q.name]
    img_path=next((v.resolve() for v in candidates if v.is_file()),None)
    if img_path is None: raise FileNotFoundError(rel)
    items.append({'id':len(items),'path':str(img_path),'gt':gt})
assert len(items)==4789, f'LTB expected 4789, received {len(items)}'
assert len({x['path'] for x in items})==len(items), 'LTB contains duplicate image paths: cannot run paired benchmark'
assert len({x['id'] for x in items})==len(items), 'LTB has duplicate sample IDs'
counts={b:sum(('26-35' if len(LT.normalize_ltb(x['gt']))<=35 else
               '36-55' if len(LT.normalize_ltb(x['gt']))<=55 else '56+')==b for x in items)
        for b in ('26-35','36-55','56+')}
assert counts=={'26-35':3376,'36-55':1147,'56+':266},counts

print('Full LTB ready:',len(items),'buckets:',counts)

In [ ]:
from pathlib import Path
Path('sage_gpu_adapter.py').write_text('"""GT-blind GPU candidate adapter for SAGE-STR v2; frozen SVTRv2-T only.\n\nAccepts PIL images only. No method here accepts image id, dataset name, labels,\nreference predictions, or ground truth.\n"""\nfrom __future__ import annotations\nfrom dataclasses import replace\nfrom time import perf_counter\nimport numpy as np\nfrom PIL import Image\nimport longtext_final as LT\nimport longtext_v3 as V3\nimport orientation_probe as OP\nimport density_probe as DP\nimport unified_router as UNI\nfrom sage_str_v2.selector import Hypothesis, Observation\nfrom sage_str_v2.ctc_evidence import from_logits\n\nclass SAGEGPUAdapter:\n    def __init__(self,net,chars,device=\'cuda\',batch_size=2):\n        self.chars=list(chars)\n        self.base=LT.LongTextRecognizer(net,self.chars,device=device,batch_size=batch_size)\n        self.dense=DP.Probe(net,self.chars,device=device,batch_size=batch_size)\n        self.unified=UNI.UnifiedRecognizer(net,self.chars,device=device,batch_size=batch_size)\n\n    def _crop_pass(self,img,parts,method,factor=1):\n        """Return stitched text, hypothesis, and agreement; no labels consumed."""\n        if factor==1:\n            logits=self.base._forward_grouped([p for p,_,_ in parts],max_ratio=12)\n        else:\n            tensors=[DP.resized_tensor(p,factor) for p,_,_ in parts]\n            logits=self.dense._forward_tensors(tensors)\n        spans=[(a,b) for _,a,b in parts]\n        events=[V3.ctc_events(lg,self.chars,a,b) for lg,(a,b) in zip(logits,spans)]\n        if len(logits)==1:\n            text=LT.normalize_ltb(LT.greedy_ctc(logits[0],self.chars))\n            agreement=0.\n        else:\n            text=V3.stitch_evidence(events,spans,use_confidence=False,geometric_fallback=True)\n            agreement=OP.overlap_agreement(events,spans)\n        original_v1_conf=OP.event_confidence(events)\n        # Per-crop CTC feature extraction. CTC frames from overlapping crops are\n        # NOT concatenated as if they were a single aligned image.\n        features=[from_logits(z,self.chars,method) for z in logits]\n        weights=np.array([len(z) for z in logits],dtype=float)\n        weights=weights/weights.sum()\n        def avg(k):return float(sum(w*getattr(f,k) for w,f in zip(weights,features)))\n        nw=LT.target_size_ltb(parts[0][0].width,parts[0][0].height,max_ratio=12)[0]\n        extra_ratio=float(max(1.,factor))\n        hyp=Hypothesis(method,text,ctc_T=sum(len(z) for z in logits),\n           confidence=avg(\'confidence\'),entropy=avg(\'entropy\'),\n           char_margin=avg(\'char_margin\'),edge_occupancy=avg(\'edge_occupancy\'),\n           overlap_agreement=agreement,n_windows=len(logits),\n           distortion=extra_ratio,cost_ms=0.)\n        return text,hyp,agreement,original_v1_conf\n\n    def infer_one(self,image):\n        img=image.convert(\'RGB\')\n        t0=perf_counter()\n        orig_logits=self.base._forward_grouped([img],max_ratio=12)[0]\n        orig=from_logits(orig_logits,self.chars,\'original\')\n        pred=LT.normalize_ltb(LT.greedy_ctc(orig_logits,self.chars))\n        assert orig.normalized==pred\n        parts=LT.make_chunks(img,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)\n        n_orig=len(parts)\n        candidates=[]\n        v3=pred\n        rot_results={}\n        dense3=None\n\n        def add_candidate(name,parts,fac=1,rot_key=None):\n            nonlocal dense3\n            started=perf_counter()\n            try:\n                text,hyp,agree,original_v1_conf=self._crop_pass(img,parts,name,fac)\n            except ValueError as exc:\n                if \'Expected an expanded width\' in str(exc):return None\n                raise\n            cost=(perf_counter()-started)*1000\n            hyp=replace(hyp,cost_ms=cost)\n            if name==\'geometry_v3\':pass\n            if name==\'crop3_width3\':dense3=text\n            if rot_key is not None:\n                rot_results[rot_key]={\'pred\':text,\'agreement\':agree,\n                                      \'confidence\':original_v1_conf}\n            if hyp.normalized!=orig.normalized:candidates.append(hyp)\n            return text\n\n        if n_orig>1:\n            v3=add_candidate(\'geometry_v3\',parts)\n        if img.width<=img.height:\n            for angle,method in [(\'cw\',\'rotate_cw\'),(\'ccw\',\'rotate_ccw\')]:\n                oriented=OP.rotated(img,angle)\n                rotated_parts=LT.make_chunks(oriented,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)\n                add_candidate(method,rotated_parts,fac=1,rot_key=angle)\n            assert set(rot_results)=={\'cw\',\'ccw\'}\n            _,choice=OP.choose_orientation(rot_results[\'cw\'],rot_results[\'ccw\'],\'overlap\')\n            v1=choice[\'pred\'];route=\'orientation\'\n        elif n_orig==1:\n            # Deliberately include v1\'s aggressive Dense proposal in the\n            # experimental candidate pool, even on short words: learned gate\n            # must detect and reject its regressions, not hide them via GT.\n            c3=DP.forced_crops(img,3)\n            add_candidate(\'crop3_width3\',c3,fac=3)\n            v1=dense3;route=\'dense\'\n            # MSR-conserving crop is a useful ablation for underread lines.\n            # This CTC-only probe does not force final adoption.\n            if len(pred)>=8:\n                add_candidate(\'crop3_msr\',c3,fac=1)\n                add_candidate(\'crop3_width2\',c3,fac=2)\n        else:\n            v1=v3;route=\'v3\'\n        # W2 is inexpensive compared with forced crops; run when the baseline\n        # predicts at least 8 chars or exhibits high CTC slot pressure.\n        if (len(pred)>=8 or orig.pressure>=.4) and img.width>img.height:\n            full=[(img,0,img.width)]\n            add_candidate(\'full_width2\',full,fac=2)\n        obs=Observation(img.width,img.height,orig,tuple(candidates))\n        assert all(c.strategy!=\'original\' for c in obs.alternatives)\n        return {\'svtrv2\':pred,\'geometry_v3\':v3,\'unified_v1\':v1,\n                \'route_v1\':route,\'observation\':obs,\n                \'candidate_time_ms\':(perf_counter()-t0)*1000}\n\n    def infer_many(self,images):\n        return [self.infer_one(im) for im in images]\n',encoding='utf-8')
import sage_gpu_adapter as SGA
print('GPU adapter ready')

In [ ]:
from PIL import Image
class Dummy(torch.nn.Module):
    def forward(self,x):
        b,_,h,w=x.shape
        t=max(2,w//4)
        z=torch.zeros((b,t,3),device=x.device)
        z[:,:,0]=1.8
        z[:,t//3,1]=3.0
        return z
adapter_test=SGA.SAGEGPUAdapter(Dummy(),['blank','a','b'],'cpu',batch_size=2)
test_images=[Image.new('RGB',(28,210),'white'),Image.new('RGB',(160,28),'white'),Image.new('RGB',(330,20),'white')]
tests=adapter_test.infer_many(test_images)
assert len(tests)==3
assert tests[0]['unified_v1']==adapter_test.unified.infer_batch([test_images[0]])[0][0]['pred']
assert all(isinstance(x['observation'],Observation) for x in tests)
assert all(x['observation'].original.normalized==x['svtrv2'] for x in tests)
print('CPU smoke tests PASS: original, rotation, Dense, geometry, and v1 control.')

In [ ]:
assert torch.cuda.is_available()
net,chars=LT.load_official_svtr(REPO,CFG,CKPT,DEVICE)
model_adapter=SGA.SAGEGPUAdapter(net,chars,DEVICE,batch_size=IMAGE_BATCH)
print('Frozen official model loaded. Classes:',len(chars))

In [ ]:
from contextlib import closing
import lmdb
from tqdm.auto import tqdm

def hash_key(source,index):
    return hashlib.sha256(f'sage-v2-fixed-v1:{source}:{index}'.encode()).hexdigest()
def selected_ids(source,total,limit):
    ids=list(range(1,total+1))
    if limit and limit<len(ids):ids=sorted(ids,key=lambda i:hash_key(source,i))[:limit]
    return sorted(ids)

COHORTS={}
COHORTS['LTB']=[{'dataset':'LTB','index':v['id'],'gt':v['gt'],'path':v['path']} for v in items]
for ds,path in SOURCES.items():
    with closing(lmdb.open(str(path.parent),readonly=True,lock=False,readahead=False,max_readers=32)) as env:
        with env.begin() as txn: total=int(txn.get(b'num-samples').decode())
    COHORTS[ds]=[{'dataset':ds,'index':i,'path':str(path)} for i in selected_ids(ds,total,MAX_EXTERNAL)]
if MAX_LTB:COHORTS['LTB']=sorted(COHORTS['LTB'],key=lambda x:hash_key('LTB',x['index']))[:MAX_LTB]
COHORTS['LTB']=sorted(COHORTS['LTB'],key=lambda x:x['index'])
print('Selected cohort counts:',{k:len(v) for k,v in COHORTS.items()})

# Disk cache of inference only; labels are stored separately in the output rows.
# RESUME does not change candidate selection.
CACHE=OUT/'inference_cache.jsonl'
LOCK=OUT/'cache_lock.json'
lock_data={'checkpoint_sha256':provenance['checkpoint_sha256'],
           'config_sha256':provenance['config_sha256'],'dataset_revision':DATA_REVISION,
           'adapter_sha256':hashlib.sha256(Path('sage_gpu_adapter.py').read_bytes()).hexdigest(),
           'selector_sha256':hashlib.sha256(Path('sage_str_v2/selector.py').read_bytes()).hexdigest()}
if not RESUME and CACHE.exists():CACHE.unlink()
if RESUME and CACHE.exists():
    if not LOCK.exists() or json.loads(LOCK.read_text())!=lock_data:
        raise RuntimeError('Old inference cache does not match checkpoint/code/dataset; remove SAGE_STR_V4_GPU_RESULTS/inference_cache.jsonl and its cache_lock.json')
LOCK.write_text(json.dumps(lock_data,indent=2))
known={}
if RESUME and CACHE.exists():
    for line in CACHE.read_text().splitlines():
        x=json.loads(line)
        known[(x['dataset'],int(x['index']))]=x
    print('Resume candidates:',len(known))

def to_save(obs):
    from dataclasses import asdict
    return {'width':obs.width,'height':obs.height,'original':asdict(obs.original),
            'alternatives':[asdict(z) for z in obs.alternatives]}
def from_save(x):
    return Observation(int(x['width']),int(x['height']),Hypothesis(**x['original']),
                       tuple(Hypothesis(**a) for a in x['alternatives']))

INFER_ROWS=[]
start=time.perf_counter()
for source,records in COHORTS.items():
    if source=='LTB':
        def load_batch(batch):
            imgs=[];gts=[]
            for r in batch:
                with Image.open(r['path']) as im:imgs.append(im.convert('RGB'))
                gts.append(r['gt'])
            return imgs,gts
        env=None
    else:
        env=lmdb.open(str(SOURCES[source].parent),readonly=True,lock=False,readahead=False,max_readers=32)
        def load_batch(batch):
            imgs=[];gts=[]
            with env.begin(buffers=False) as txn:
                for r in batch:
                    idx=int(r['index']); key=f'{idx:09d}'.encode()
                    data=txn.get(b'image-'+key);label=txn.get(b'label-'+key)
                    if data is None or label is None:raise RuntimeError((source,idx))
                    with Image.open(io.BytesIO(data)) as im:imgs.append(im.convert('RGB'))
                    gts.append(label.decode('utf-8',errors='replace'))
            return imgs,gts
    try:
        for j in tqdm(range(0,len(records),IMAGE_BATCH),desc=f'SAGE inference {source}'):
            batch=records[j:j+IMAGE_BATCH]
            if all((source,int(r['index'])) in known for r in batch):
                INFER_ROWS.extend(known[(source,int(r['index']))] for r in batch)
                continue
            images,gts=load_batch(batch)
            runs=model_adapter.infer_many(images)
            assert len(runs)==len(batch)
            # Spot-check that the Unified v1 control is EXACTLY its original adapter,
            # including CW/CCW tie-breaking and crop3 width expansion.
            if j==0:
                check,_=model_adapter.unified.infer_batch(images)
                assert all(x['unified_v1']==ref['pred'] for x,ref in zip(runs,check)), (source,'Unified v1 parity mismatch')
            for r,gt,run in zip(batch,gts,runs):
                out={'dataset':source,'index':int(r['index']),'gt':str(gt),'svtrv2':run['svtrv2'],
                     'geometry_v3':run['geometry_v3'],'unified_v1':run['unified_v1'],
                     'router_v1':run['route_v1'],'observation':to_save(run['observation']),
                     'gpu_candidates':len(run['observation'].alternatives)}
                known[(source,int(r['index']))]=out;INFER_ROWS.append(out)
                with open(CACHE,'a',encoding='utf-8') as f:f.write(json.dumps(out,ensure_ascii=False)+'\n')
    finally:
        if env is not None:env.close()
assert len(INFER_ROWS)==sum(len(x) for x in COHORTS.values())
assert len({(x['dataset'],x['index']) for x in INFER_ROWS})==len(INFER_ROWS)
(INFER_ROWS and OUT.joinpath('inference_complete.marker').write_text(str(len(INFER_ROWS))))
print('Inference complete',len(INFER_ROWS),' seconds',round(time.perf_counter()-start,1))

In [ ]:
# 5. Stable identity-based split; same exact candidate evidence for v3 and v4.
from sage_str_v3.selector import DualArbiter, reference
from sage_str_v4.selector import ContextualDualArbiter

SPLIT_SALT='cs406-sage-v4-locked-splits-v1'
SPLIT_LIMITS=(.44,.55,.66,.80,1.0)
SPLIT_NAMES=('train','calibration','tune','audit','eval')
def split_for(dataset,index):
    value=hashlib.sha256(f'{SPLIT_SALT}|{dataset}|{int(index)}'.encode('utf-8')).digest()
    unit=int.from_bytes(value[:8],'big')/(1<<64)
    return next(name for name,hi in zip(SPLIT_NAMES,SPLIT_LIMITS) if unit<hi)

def locked_split_demo():
    ids=[('LTB',i) for i in range(1,500)]
    m={(ds,i):split_for(ds,i) for ds,i in ids}
    for limit in (10,37,195,499):
        assert all(split_for(ds,i)==m[ds,i] for ds,i in ids[:limit])
    return True
assert locked_split_demo()
SPLITS={n:[] for n in SPLIT_NAMES}
for row in INFER_ROWS:SPLITS[split_for(row['dataset'],row['index'])].append(row)
assert sum(map(len,SPLITS.values()))==len(INFER_ROWS)
assert len({(r['dataset'],r['index']) for part in SPLITS.values() for r in part})==len(INFER_ROWS)
for ds in COHORTS:
    print('SPLIT',ds,{k:sum(r['dataset']==ds for r in v) for k,v in SPLITS.items()})
print('Locked split sizes:',{k:len(v) for k,v in SPLITS.items()})
assert all(SPLITS[k] for k in SPLIT_NAMES),'Insufficient cohort for fixed split; increase MAX_LTB/MAX_EXTERNAL'

def labeled(rows):
    return [(from_save(r['observation']),LT.normalize_ltb(r['gt']),r['dataset'])
            for r in rows if LT.normalize_ltb(r['gt'])]

def fit_policy(selector,tag):
    selector.fit(labeled(SPLITS['train']))
    cal=selector.calibrate(labeled(SPLITS['calibration']))
    tune=selector.choose_thresholds(labeled(SPLITS['tune']))
    audit=selector.audit(labeled(SPLITS['audit']),min_n=100)
    print(tag,'temperature calibration:',cal)
    print(tag,'thresholds:',selector._printable_thresholds(),' audit:',audit['passed'])
    return {'calibration':cal,'tune':tune,'audit':audit}

sage3_control=DualArbiter(harm_weight=2.5,cer_weight=1.0,cost_weight=.002)
PROTOCOL_V3=fit_policy(sage3_control,'SAGE v3 control / new locked split')
sage4=ContextualDualArbiter(harm_weight=2.5,cer_weight=1.0,cost_weight=.002)
PROTOCOL_V4=fit_policy(sage4,'SAGE v4 contextual')

from sage_str_v4.batch import score_observations
from sage_str_v3 import selector as v3mod
from sage_str_v4 import selector as v4mod
obs_list=[from_save(r['observation']) for r in INFER_ROWS]
V3_DECISIONS,V3_TRACES=score_observations(sage3_control,obs_list,v3mod)
V4_DECISIONS,V4_TRACES=score_observations(sage4,obs_list,v4mod)
LOOKUP={(r['dataset'],r['index']):i for i,r in enumerate(INFER_ROWS)}
assert len(LOOKUP)==len(INFER_ROWS)
# Per-observation and batched implementations must be decision-equivalent.
for s,mod,bulk,tag in ((sage3_control,v3mod,V3_DECISIONS,'v3'),(sage4,v4mod,V4_DECISIONS,'v4')):
    for idx in np.linspace(0,len(obs_list)-1,min(35,len(obs_list)),dtype=int):
        assert s.select(obs_list[idx])==bulk[idx],f'{tag}: batch parity failure index={idx}'
    # Perturb ordering to ensure deterministic choice is independent of batch ordering.
    indexes=list(reversed(range(min(50,len(obs_list)))))
    other,_=score_observations(s,[obs_list[j] for j in indexes],mod)
    assert other==[bulk[j] for j in indexes],f'{tag}: order-dependent choice'
    print(tag,'batched scoring parity + ordering replay PASS',len(obs_list),'observations')
for rr in V4_TRACES:
    for r in rr:
        assert abs(r['p_neither']+r['p_gain']+r['p_harm']-1.)<1e-8
(OUT/'v4_locked_splits.csv').write_text('dataset,index,split\n'+''.join(
    f"{r['dataset']},{r['index']},{name}\n" for name,rows in SPLITS.items() for r in rows))
(OUT/'v3_control_protocol.json').write_text(json.dumps(PROTOCOL_V3,indent=2,default=str))
(OUT/'v4_policy.json').write_text(json.dumps(sage4.manifest(),indent=2,default=str))
(OUT/'v4_training_protocol.json').write_text(json.dumps({
   'cohort_invariant_split':SPLIT_SALT,'limits':SPLIT_LIMITS,
   'splits':{k:len(v) for k,v in SPLITS.items()},
   'v4':PROTOCOL_V4,'warning':'All benchmark sources have been historically inspected. Development evaluation only.'
},indent=2,default=str))


In [ ]:
# 6. EVAL ONLY: same exact test images, identical input evidence, no fit/tune on this partition.
from scipy.stats import binomtest
COLS=('SVTRv2-T_single','SVTRv2-T_v3','Unified_v1','Safe_reference','SAGE_v3_locked','SAGE_v4')
MAP={'SVTRv2-T_single':'svtrv2','SVTRv2-T_v3':'geometry_v3','Unified_v1':'unified_v1',
     'Safe_reference':'reference_prediction','SAGE_v3_locked':'sage_v3','SAGE_v4':'sage_v4'}

def summary_table(rows,split):
    measurements=[];paired=[];per_image=[]
    if not rows:return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()
    for ds,g in pd.DataFrame(rows).groupby('dataset',sort=False):
        gold=[LT.normalize_ltb(v) for v in g['gt'].tolist()]
        valid=np.asarray([bool(x) for x in gold],dtype=bool)
        correct={}; edits_by_method={}
        for m in COLS:
            pred=[LT.normalize_ltb(v) for v in g[MAP[m]].tolist()]
            exact=np.asarray([a==b for a,b in zip(gold,pred)])
            edits=np.asarray([Levenshtein.distance(a,b) if a else 0 for a,b in zip(gold,pred)])
            correct[m]=exact;edits_by_method[m]=edits
            denom=sum(len(t) for t,ok in zip(gold,valid) if ok)
            measurements.append({'split':split,'dataset':ds,'method':m,
                'n':int(valid.sum()),'exact_correct':int((exact&valid).sum()),
                'exact_accuracy_pct':100*float(exact[valid].mean()) if valid.any() else float('nan'),
                'CER_pct':100*float(edits[valid].sum())/max(denom,1),
                'total_edit_errors':int(edits[valid].sum())})
        for m in COLS[:-1]:
            older=correct[m]&valid;newer=correct['SAGE_v4']&valid
            w=int((newer&~older).sum());h=int((older&~newer).sum())
            paired.append({'split':split,'dataset':ds,'comparison':'SAGE_v4 vs '+m,
                'win':w,'loss':h,'net':w-h,
                'McNemar_exact_p':float(binomtest(min(w,h),n=w+h,p=.5).pvalue) if w+h else 1.})
        for i,r in enumerate(g.itertuples(index=False)):
            out=dict(r._asdict())
            out['baseline_correct']=int(correct['SVTRv2-T_single'][i]) if valid[i] else -1
            out['unified_correct']=int(correct['Unified_v1'][i]) if valid[i] else -1
            out['v3_correct']=int(correct['SAGE_v3_locked'][i]) if valid[i] else -1
            out['v4_correct']=int(correct['SAGE_v4'][i]) if valid[i] else -1
            out['v4_minus_unified_edit_errors']=int(edits_by_method['SAGE_v4'][i]-edits_by_method['Unified_v1'][i]) if valid[i] else 0
            per_image.append(out)
    return pd.DataFrame(measurements),pd.DataFrame(paired),pd.DataFrame(per_image)

def evaluate_rows(records):
    rows=[]
    for r in records:
        o=from_save(r['observation']);j=LOOKUP[r['dataset'],r['index']]
        v3=V3_DECISIONS[j];v4=V4_DECISIONS[j]
        ref=reference(o)
        rows.append({'dataset':r['dataset'],'index':r['index'],'gt':r['gt'],
            'svtrv2':r['svtrv2'],'geometry_v3':r['geometry_v3'],
            'unified_v1':r['unified_v1'],'reference_prediction':ref.normalized,
            'reference_strategy':ref.strategy,'sage_v3':v3['prediction'],
            'sage_v3_strategy':v3['chosen'],'sage_v4':v4['prediction'],
            'sage_v4_strategy':v4['chosen'],'sage_v4_reason':v4['reason'],
            'gpu_candidates':r['gpu_candidates']})
    return rows
EVAL_PRED=evaluate_rows(SPLITS['eval'])
EV_M,EV_P,EV_R=summary_table(EVAL_PRED,'eval')
EV_M.to_csv(OUT/'evaluation_accuracy_cer_by_dataset.csv',index=False)
EV_P.to_csv(OUT/'evaluation_paired_win_loss.csv',index=False)
EV_R.to_csv(OUT/'evaluation_predictions.csv',index=False)
print('=== KAGGLE SAGE v4 / EVAL ONLY: FIXED SPLIT ===')
print(EV_M.to_string(index=False,float_format=lambda x:f'{x:.3f}'))
print('=== Paired SAGE v4 against v3 locked, Unified v1, baseline ===')
print(EV_P.to_string(index=False))
print('=== V4 final choices ===')
print(EV_R['sage_v4_strategy'].value_counts().to_string())


In [ ]:
# 7. Full eval candidate trace; GT is used POSTHOC only for failure audit.
TRACE=[]
for r in SPLITS['eval']:
    o=from_save(r['observation']);gold=LT.normalize_ltb(r['gt'])
    j=LOOKUP[r['dataset'],r['index']];v4=V4_DECISIONS[j];ref=reference(o)
    for d in V4_TRACES[j]:
        TRACE.append({'dataset':r['dataset'],'index':r['index'],
           'candidate':d['strategy'],'group':d['group'],
           'candidate_prediction':d['prediction'],'reference_prediction':ref.normalized,
           'selected_strategy':v4['chosen'],'utility':d['score'],
           'threshold':d['threshold'],'p_gain':d['p_gain'],
           'p_harm':d['p_harm'],'p_neither':d['p_neither'],
           'predicted_CER_gain':d['predicted_cer_gain'],
           'selected':d['accepted'],'candidate_correct_POSTHOC':int(d['prediction']==gold),
           'reference_correct_POSTHOC':int(ref.normalized==gold),
           'actual_char_edit_gain_POSTHOC':Levenshtein.distance(ref.normalized,gold)-Levenshtein.distance(d['prediction'],gold)})
TR=pd.DataFrame(TRACE)
TR.to_csv(OUT/'evaluation_per_candidate_trace.csv',index=False)
if not TR.empty:
    assert ((TR.p_gain+TR.p_harm+TR.p_neither-1).abs()<1e-8).all()
    missed=TR[(TR.candidate_correct_POSTHOC==1)&(TR.reference_correct_POSTHOC==0)&(~TR.selected)]
    missed.to_csv(OUT/'evaluation_rejected_correct_proposals_POSTHOC.csv',index=False)
    print('Correct proposals rejected (POSTHOC):',len(missed))
else:
    print('No changed candidates available for trace')

MO=EV_R[EV_R.dataset=='u14m/multi_oriented'].copy()
MO_LOSS=MO[(MO.unified_correct==1)&(MO.v4_correct==0)]
MO_WIN=MO[(MO.unified_correct==0)&(MO.v4_correct==1)]
MO_LOSS.to_csv(OUT/'evaluation_multi_oriented_unified_wins_v4_losses.csv',index=False)
print('=== Multi-Oriented: Unified correct, V4 wrong ===')
print(MO_LOSS[['index','gt','svtrv2','unified_v1','sage_v4','sage_v4_strategy']].to_string(index=False))
print('=== Multi-Oriented: V4 correct, Unified wrong ===')
print(MO_WIN[['index','gt','unified_v1','sage_v4','sage_v4_strategy']].to_string(index=False))
if not TR.empty and len(MO_LOSS):
    detail=TR[(TR.dataset=='u14m/multi_oriented') & TR['index'].isin(MO_LOSS['index'])]
    print('=== Multi-Oriented failed cases: candidate decision scores (POSTHOC) ===')
    print(detail[['index','candidate','candidate_prediction','utility','threshold','p_gain','p_harm','predicted_CER_gain','candidate_correct_POSTHOC']].to_string(index=False))

LTB_AUDIT=EV_R[EV_R.dataset=='LTB'].copy()
LTB_AUDIT.sort_values('v4_minus_unified_edit_errors',ascending=False).to_csv(OUT/'evaluation_ltb_error_audit.csv',index=False)
print('=== LTB worst character regressions vs Unified (POSTHOC) ===')
print(LTB_AUDIT.nlargest(8,'v4_minus_unified_edit_errors')[['index','gt','unified_v1','sage_v4','sage_v4_strategy','v4_minus_unified_edit_errors']].to_string(index=False))


In [ ]:
# 8. All-cohort DEVELOPMENT-DESCRIPTIVE metrics (not a held-out evaluation).
ALL_PRED=evaluate_rows(INFER_ROWS)
ALL_M,_,ALL_R=summary_table(ALL_PRED,'all_descriptive')
ALL_M.to_csv(OUT/'all_descriptive_accuracy_cer.csv',index=False)
ALL_R.to_csv(OUT/'all_descriptive_predictions.csv',index=False)
BKT=[]
for bucket,(lo,hi) in {'26-35':(26,35),'36-55':(36,55),'56+':(56,10**8)}.items():
    g=[r for r in EVAL_PRED if r['dataset']=='LTB' and lo<=len(LT.normalize_ltb(r['gt']))<=hi]
    for m,col in [('SVTRv2-T_single','svtrv2'),('Unified_v1','unified_v1'),
                  ('Safe_reference','reference_prediction'),('SAGE_v3_locked','sage_v3'),('SAGE_v4','sage_v4')]:
        n=len(g);correct=sum(LT.normalize_ltb(r[col])==LT.normalize_ltb(r['gt']) for r in g)
        BKT.append({'bucket':bucket,'method':m,'n':n,'exact_correct':correct,
                    'exact_accuracy_pct':100*correct/n if n else np.nan})
BKT=pd.DataFrame(BKT);BKT.to_csv(OUT/'ltb_eval_length_buckets.csv',index=False)
AAVG=[]
for m,g in BKT.groupby('method'):
    if len(g)==3 and (g.n>0).all():AAVG.append({'method':m,'eval_AAvg_pct':float(g.exact_accuracy_pct.mean()),'n_LTB':sum(g.n)})
AAVG=pd.DataFrame(AAVG);AAVG.to_csv(OUT/'ltb_eval_AAvg.csv',index=False)
print('=== LTB bucket evaluation ===')
print(BKT.to_string(index=False,float_format=lambda x:f'{x:.3f}'))
print('=== LTB A-Avg ===')
print(AAVG.to_string(index=False))
print('=== All cohort, DEVELOPMENT only ===')
print(ALL_M.to_string(index=False,float_format=lambda x:f'{x:.3f}'))


In [ ]:
# 8b. Predeclared exploratory acceptance checks (NOT an independent test).
# No threshold is re-fit or changed based on any evaluation rows.
CHECKS=[]
def val(ds,method,key):
    r=EV_M[(EV_M['dataset']==ds)&(EV_M['method']==method)]
    return float(r.iloc[0][key]) if len(r)==1 else float('nan')

for ds in ('evaluation/IIIT5k','evaluation/CUTE80'):
    CHECKS.append({'criterion':ds+' exact >= SVTRv2-T original',
        'passed':bool(val(ds,'SAGE_v4','exact_correct')>=val(ds,'SVTRv2-T_single','exact_correct'))})
    CHECKS.append({'criterion':ds+' CER <= SVTRv2-T original',
        'passed':bool(val(ds,'SAGE_v4','CER_pct')<=val(ds,'SVTRv2-T_single','CER_pct')+1e-9)})

ds='u14m/multi_oriented'
CHECKS.append({'criterion':'Multi-Oriented exact >= Unified v1',
    'passed':bool(val(ds,'SAGE_v4','exact_correct')>=val(ds,'Unified_v1','exact_correct'))})
CHECKS.append({'criterion':'Multi-Oriented CER <= SVTRv2-T original',
    'passed':bool(val(ds,'SAGE_v4','CER_pct')<=val(ds,'SVTRv2-T_single','CER_pct')+1e-9)})

ds='u14m/multi_words'
CHECKS.append({'criterion':'Multi-Words exact >= SVTRv2-T original',
    'passed':bool(val(ds,'SAGE_v4','exact_correct')>=val(ds,'SVTRv2-T_single','exact_correct'))})

ds='LTB'
CHECKS.append({'criterion':'LTB exact >= Unified v1',
    'passed':bool(val(ds,'SAGE_v4','exact_correct')>=val(ds,'Unified_v1','exact_correct'))})
CHECKS.append({'criterion':'LTB CER <= Unified v1',
    'passed':bool(val(ds,'SAGE_v4','CER_pct')<=val(ds,'Unified_v1','CER_pct')+1e-9)})
for bucket in ('26-35','36-55','56+'):
    a=BKT[(BKT.bucket==bucket)&(BKT.method=='SAGE_v4')]
    b=BKT[(BKT.bucket==bucket)&(BKT.method=='Unified_v1')]
    CHECKS.append({'criterion':'LTB '+bucket+' exact >= Unified v1',
        'passed':bool(len(a)==1 and len(b)==1 and a.iloc[0]['n']>0 and
                      a.iloc[0]['exact_correct']>=b.iloc[0]['exact_correct'])})
ACCEPTANCE=pd.DataFrame(CHECKS)
ACCEPTANCE.to_csv(OUT/'evaluation_exploratory_acceptance.csv',index=False)
print('=== EXPLORATORY ACCEPTANCE (not OOD evidence) ===')
print(ACCEPTANCE.to_string(index=False))
print('ALL CHECKS:',bool(ACCEPTANCE.passed.all()),'— if False, do not claim v4 solves all failures.')


In [ ]:
# 9. Kaggle-native artifacts. Save Version -> Save & Run All to persist Output.
import datetime,joblib,zipfile
V4_CODE_SHA=hashlib.sha256((WORK/'sage_str_v4/selector.py').read_bytes()).hexdigest()
for name,model in (('v3_control_locked_arbiter.joblib',sage3_control),('v4_contextual_arbiter.joblib',sage4)):
    out_path=OUT/name
    joblib.dump(model,out_path)
    loaded=joblib.load(out_path)
    indexes=np.linspace(0,len(obs_list)-1,min(50,len(obs_list)),dtype=int)
    assert [loaded.select(obs_list[j]) for j in indexes]==[model.select(obs_list[j]) for j in indexes]
print('Model serialization replay PASS')

REPORT=['# SAGE-STR v4 — Kaggle GPU exploratory development report','',
 f'UTC: {datetime.datetime.now(datetime.timezone.utc).isoformat()}',
 f'OpenOCR commit: {OPENOCR_REV}',f'Checkpoint SHA256: {provenance["checkpoint_sha256"]}',
 f'V4 code SHA256: {V4_CODE_SHA}',
 f'Fixed split salt: {SPLIT_SALT}',
 f'MAX_LTB={MAX_LTB}, MAX_EXTERNAL={MAX_EXTERNAL}; 0 means full cohort',
 'Both v3 and v4 are re-fitted using identical fixed partitions and same GPU predictions.',
 '', '## EVAL-ONLY (within-run disjoint, but historical benchmarks inspected)',
 EV_M.to_markdown(index=False,floatfmt='.3f'),'',
 '## Paired test',EV_P.to_markdown(index=False,floatfmt='.3f'),'',
 '## LTB buckets',BKT.to_markdown(index=False,floatfmt='.3f'),'',
 '## V4 safety audit',str(PROTOCOL_V4['audit']),'',
 '## Limitations',
 '- This is a development evaluation. Previously studied benchmark images are NOT independent held-out generalization evidence.',
 '- No threshold, candidate selection or feature is permitted to read eval ground-truth at inference.',
 '- The stable hash split changes the rows compared with the historical v3 pilot; only within-notebook paired v3_locked vs v4 comparison is valid.',
 '- Safety audit is empirical only, NOT guaranteed OOD non-inferiority or conformal validity.',
 '- The official SVTRv2-T backbone is FROZEN; only lightweight calibration/selector is learned.',
 '- Candidate pool is the same as v3; gains are due to the decision layer, not new OCR models.',
 '- To make final conclusions, evaluate a truly untouched long-text and rotated-word benchmark.'
]
(OUT/'REPORT_SAGE_V4.md').write_text('\n'.join(REPORT),encoding='utf-8')
(OUT/'run_manifest.json').write_text(json.dumps({
    'model':'frozen official SVTRv2-T','openocr_rev':OPENOCR_REV,
    'checkpoint_sha256':provenance['checkpoint_sha256'],'config_sha256':provenance['config_sha256'],
    'dataset_revision':DATA_REVISION,'v4_code_sha256':V4_CODE_SHA,
    'split_salt':SPLIT_SALT,'split_sizes':{k:len(v) for k,v in SPLITS.items()},
    'inference_rows':len(INFER_ROWS),'eval_rows':len(SPLITS['eval']),
    'v4_audit_passed':sage4.audited,'v3_audit_passed':sage3_control.audited,
    'historical_benchmarks_used_for_development':True,
},indent=2))
archive=WORK/'SAGE_STR_V4_GPU_RESULTS.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in OUT.iterdir():
        if p.is_file() and p!=CACHE:z.write(p,p.name)
    for path in ('sage_str_v4/selector.py','sage_str_v4/batch.py','sage_str_v3/selector.py','sage_gpu_adapter.py'):
        z.write(WORK/path,'source/'+path)
print('CSV/REPORT SAVED to:',OUT)
print('ZIP (optional, no download required):',archive,'size',archive.stat().st_size)
print('Available outputs:')
for p in sorted(OUT.iterdir()):
    if p.is_file():print(' ',p.name,round(p.stat().st_size/1024,1),'KiB')
print('Kaggle: use Save Version -> save output or inspect printed eval + forensic tables.')


## How to interpret v4

- Primary: `evaluation_accuracy_cer_by_dataset.csv` + `evaluation_paired_win_loss.csv`. The `SAGE_v3_locked` control is retrained on exactly the **same stable partition** as v4 (not historical v3 scores).
- Check whether v4 **retains** IIIT5K/CUTE80, improves Multi-Oriented relative to **Unified v1**, and reduces LTB exact/CER errors. If it does not, **report failure**, do not alter thresholds using these eval predictions.
- `evaluation_per_candidate_trace.csv` includes `p_gain`, `p_harm`, `p_neither`, utility/threshold, predictions and posthoc correct flags for auditing.
- `evaluation_multi_oriented_unified_wins_v4_losses.csv` lists rotated-word failures; `evaluation_ltb_error_audit.csv` lists largest character regressions vs Unified.
- All CSV, fitted selector, source, hashes, report, and optional ZIP remain at `/kaggle/working/SAGE_STR_V4_GPU_RESULTS`. For persistent Kaggle storage, **Save Version → Save & Run All**.
- `MAX_EXTERNAL=0`, `MAX_LTB=0` runs complete cohorts. The split remains fixed for every image ID, unlike earlier notebooks. However historical datasets have been repeatedly inspected; full run is still development evidence.
